# Prédiction du prochain `cell_id` — une semaine, sept entraînements

Variante « semaine » de `train_cellid_llm.ipynb`. Au lieu d'un entraînement unique,
ce notebook enchaîne **un entraînement indépendant par jour** (lundi → dimanche) sur
sept jeux de données distincts, puis agrège tout pour les comparer.

## Ce qui change par rapport au notebook d'origine, et pourquoi

| # | Changement | Raison |
|---|---|---|
| 1 | Tokenizer et vocabulaire `cell_id` construits **une seule fois** sur l'union des 7 jours | Les 7 softmax ont la même taille → les `top-k` sont comparables entre jours. Sur ce jeu : 369 `cell_id` par jour, 369 en union — le partage ne coûte donc quasiment rien. |
| 2 | Modèle de base **rechargé** à chaque jour + adaptateur LoRA neuf | Aucun poids ne fuit d'un jour sur l'autre. `get_peft_model` enveloppe le modèle *en place* : le rappeler sur un modèle déjà enveloppé empilerait les adaptateurs. |
| 3 | Tout l'état par-jour vit dans un objet `DayContext` | Le notebook d'origine passe par des globales (`users_train`, `GROUP_MODEL`, …) que `evaluate_cell_accuracy` lit implicitement. En boucle, oublier d'en rebinder une valide le mardi sur les utilisateurs du lundi **sans erreur visible**. |
| 4 | Boucle **reprenable** : un jour terminé écrit son JSON et est sauté au relancement | Colab déconnecte. Une coupure coûte le jour en cours, pas les sept. |
| 5 | Évaluation **batchée** : un forward par batch de 8 séquences au lieu d'un par utilisateur | L'éval faisait 1900 forwards unitaires à chaque epoch : c'était la moitié du temps de calcul. Le score est **identique** (padding à droite + `attention_mask` : en attention causale les positions réelles ne voient jamais le padding). |
| 6 | `save_strategy="no"`, seul l'adaptateur LoRA (~30 Mo) est écrit en fin de jour | Le notebook d'origine écrivait ~550 Mo à chaque amélioration de validation. Les meilleurs poids sont déjà gardés en RAM par le callback. |

**Tous les hyperparamètres d'entraînement sont ceux du notebook d'origine** —
`max_epochs`, patience, batch, learning rate, LoRA, chunking, `context_fraction`.
Les six changements ci-dessus portent sur l'orchestration, l'isolation et
l'évaluation, jamais sur l'optimiseur. Chaque jour s'entraîne donc dans le même
régime que vos runs précédents. La section « Ce qu'il ne faut PAS régler pour
aller plus vite » des Notes explique pourquoi.

## Attention à la lecture des résultats

Les sept jeux de données n'ont **aucun utilisateur en commun** (vérifié : intersection
vide entre tous les couples de jours). Un écart entre deux jours mélange donc *effet
jour de la semaine* et *effet échantillon d'utilisateurs*. C'est pourquoi tout est
rapporté **relativement à la baseline de persistance du jour** (« répéter le dernier
`cell_id` »), qui mesure la difficulté intrinsèque de l'échantillon : la colonne à
regarder pour comparer les jours est `écart`, pas `top1_seen`.

## Utilisation sur Colab

1. Uploader les 14 fichiers JSONL (7 jours × train/test) dans `/content` (ou un
   dossier de Drive), puis renseigner `WEEK_CONFIG["data_dirs"]` si besoin.
2. Exécuter toutes les cellules. La cellule BOUCLE peut être relancée telle quelle
   après une déconnexion : elle reprend là où elle s'était arrêtée.
3. Les cellules SYNTHÈSE et GRAPHIQUES lisent les JSON écrits par la boucle — elles
   fonctionnent même sur une session neuve, sans refaire les entraînements.

In [ ]:
# ============================== CONFIGURATION ==============================
from pathlib import Path
import sys

# Ajout des dossiers "python" et répertoires courants au PYTHONPATH
for _p in ("python", ".", "/content", "/content/python"):
    _path = str(Path(_p).resolve())
    if Path(_p).exists() and _path not in sys.path:
        sys.path.insert(0, _path)

# --- DÉBUT DU BLOC GÉNÉRÉ (python/sync_notebook_fallbacks.py) ---
# ⚠️  NE PAS ÉDITER À LA MAIN : bloc régénéré par `make sync-notebook`.
# Chaque module de python/ est embarqué ici en clair pour que le notebook soit
# AUTONOME (Colab neuf, machine vierge : aucun fichier annexe à uploader). Le
# notebook n'écrit un module sur le disque que si son import échoue -- une
# installation normale du dépôt continue donc d'utiliser python/<module>.py.
# `tests/test_notebook_fallbacks.py` échoue si une copie ci-dessous diverge de
# son fichier source, pour qu'il n'existe qu'une seule source de vérité.
_EMBEDDED_MODULES = {
    'cellid_encoding': r'''"""Time-aware event encoding shared by the data-prep scripts and the training
notebook.

An "event" is one (cell_id, hour) pair from a user's daily trajectory. Hour is
the hour-of-day (0-23) derived from the raw timestamp (seconds since local
midnight, per CLAUDE.md) attached to that cell_id -- the timestamp follows the
cell_id it belongs to.
"""


def event_hour_from_seconds(ts_seconds):
    """Hour of day (0-23) from a timestamp in seconds since local midnight."""
    return (int(ts_seconds) // 3600) % 24


def hour_token(hour):
    """Dedicated tokenizer token for an hour-of-day, e.g. hour_token(6) == '<H06>'."""
    return f"<H{hour:02d}>"


HOUR_VOCAB = [hour_token(h) for h in range(24)]


def in_transition_window(hour, transition_windows):
    """True if `hour` falls in any (start_hour, end_hour, weight) window.
    Bounds are half-open [start_hour, end_hour) -- end excluded."""
    return any(start <= hour < end for start, end, _weight in transition_windows)


def event_weight(hour, transition_windows, base_weight=1.0):
    """Loss weight for an event at `hour`: the configured window's weight if
    `hour` falls inside one of `transition_windows`, else `base_weight`.
    `hour=None` (hour-less legacy data) always returns `base_weight`."""
    if hour is None:
        return base_weight
    for start, end, weight in transition_windows:
        if start <= hour < end:
            return weight
    return base_weight


def split_index(n_events, context_fraction):
    """Cutoff index for a context/target split over `n_events` events: at
    least 1, at most n_events - 1."""
    return max(1, min(n_events - 1, round(n_events * context_fraction)))


def make_chunks(n_events, chunk_len, chunk_stride):
    """Sliding-window (start, end, n_context_events) triples over event
    indices [0, n_events). `n_context_events` is how many of the window's
    leading events are overlap from the previous window (already trained on,
    excluded from the loss the second time). Windows are at most `chunk_len`
    events, advancing by `chunk_stride` each time."""
    if n_events <= chunk_len:
        return [(0, n_events, 0)]
    out, start = [], 0
    while start < n_events:
        end = min(start + chunk_len, n_events)
        ctx = 0 if start == 0 else chunk_len - chunk_stride
        if end - start > ctx:
            out.append((start, end, ctx))
        if start + chunk_len >= n_events:
            break
        start += chunk_stride
    return out


def event_token_positions(n_events, has_hours):
    """Index (relative to the first token *after* the prompt prefix) of each
    event's cell_id token in the flat sequence built by `encode_events`. When
    `has_hours` is True, each event is 2 tokens (hour then cell_id), so
    cell_id tokens sit at 1, 3, 5, ...; when False, each event is 1 token
    (cell_id only), so they sit at 0, 1, 2, ..."""
    step = 2 if has_hours else 1
    offset = 1 if has_hours else 0
    return [offset + i * step for i in range(n_events)]


def encode_events(cells, hours, cell_to_id, hour_to_id, prefix_ids,
                  transition_windows, n_masked_cells=0, base_weight=1.0):
    """Build (input_ids, labels, weights) for one sequence of events.

    cells: list[str] cell_id per event.
    hours: list[int] | None. If None, cell tokens only (legacy/no hour info) --
        no interleaved hour tokens, every unmasked position gets `base_weight`.
    cell_to_id / hour_to_id: dict[str, int] tokenizer vocab lookups.
    prefix_ids: list[int] tokens prepended before any event (e.g. a fixed
        prompt) -- always masked (label -100, weight base_weight).
    n_masked_cells: the first N events' cell labels are masked (-100) --
        used for chunk overlap, so a repeated context window isn't trained on
        twice.
    Returns three lists of equal length (one entry per token): input_ids,
    labels, weights.
    """
    ids = list(prefix_ids)
    labels = [-100] * len(prefix_ids)
    weights = [base_weight] * len(prefix_ids)
    for i, cell in enumerate(cells):
        hour = hours[i] if hours is not None else None
        if hour is not None:
            ids.append(hour_to_id[hour_token(hour)])
            labels.append(-100)
            weights.append(base_weight)
        ids.append(cell_to_id[cell])
        if i < n_masked_cells:
            labels.append(-100)
            weights.append(base_weight)
        else:
            labels.append(cell_to_id[cell])
            weights.append(event_weight(hour, transition_windows, base_weight))
    return ids, labels, weights
''',
    'user_groups': r'''"""Behavioural user groups for cross-training.

Rationale
---------
`cell_id` sequences look very different from one user to the next, but the
differences are not arbitrary: they are largely captured by *how often the user
changes physical site, as a function of the hour of day*. Two users who both
"stay put in the morning and move a lot late in the afternoon" are far more
informative about each other than two users picked at random -- which is
exactly what cross-training needs.

A group is therefore defined by a **site-change-rate profile over hour bands**:

    profile[b] = P(site root changes | the event falls in band b)

Measuring *site root* changes rather than *cell_id* changes matters. Per
CLAUDE.md a `cell_id` is `<tech letter><site root><zone digits>`, so the same
physical antenna appears under several `cell_id` values. On this dataset 12.4%
of all consecutive-event transitions are "same site root, different
radio/sector" -- i.e. the phone re-attached to another cell of the *same*
antenna without the user going anywhere. Counting those as movement would blur
the very distinction the groups are meant to capture, so `site_root()` strips
the technology letter and the zone digits before comparing.

Sparse bands are handled by shrinking each user's per-band rate toward the
global (train-set) rate for that band, with `shrinkage` pseudo-transitions of
prior weight -- a user with 3 night events does not get a wildly confident
night rate.

What the analysis found on 400_users_train.jsonl (4 bands, k=4)
---------------------------------------------------------------
Groups are numbered by increasing overall mobility, so the ordering is stable
and meaningful (G0 = most sedentary, G3 = most mobile in the morning):

    G0 "sédentaire"        28% of users  site-change 28/32/34/38% (sleep/morn/day/eve)
                           persistence 56% -- "repeat the last cell_id" is already strong
    G1 "régulier"          35%           31/47/46/45% -- flat, moderate mobility
                           persistence 42%
    G2 "actif après-midi"  14%           30/47/67/60% -- calm morning, very mobile
                           persistence 20%    from 16h to 20h (peak change rate 84%)
    G3 "actif le matin"    23%           35/67/65/43% -- very mobile 07h-13h, then
                           persistence 18%    settles down in the evening

G2 and G3 are near mirror images in time, and that shape is not an artefact of
sequence length: the (morning - evening) rate asymmetry is +0.244 for G3 vs
-0.135 for G2 while correlating only -0.28 with log(sequence length).

Why this changes the loss weighting
-----------------------------------
The single global transition window `(4, 6)` turns out to be *easier* than
average for all four groups (persistence 49-64% inside it vs 20-57% overall) --
so upweighting it was pushing gradient at positions the baseline already gets
right. And `(18, 20)` is only genuinely hard for G2; for G3 it is where the
user has just settled down for the evening (36% persistence vs 20% overall,
i.e. easier). `derive_group_windows()` replaces that one global guess with
per-group windows read off the data: the contiguous hours where a group's
`cell_id` persistence falls furthest below its own average, which is precisely
where the model has to reason instead of repeating.

This is the "keyed by group rather than a single global default" iteration that
CLAUDE.md anticipated.
"""
import re
from collections import Counter

import numpy as np

# ---------------------------------------------------------------- cell parsing
# <tech letter><site root><zone digits>; zone is 3 digits for 3G (<1|2><01-09>)
# and a single digit for 2G. See CLAUDE.md.
CELL_ID_RE = re.compile(r"^([BDUV])([A-Z]+?)((?:[12]\d{2})|\d)$")


def parse_cell_id(cell_id):
    """(tech_letter, site_root, zone_digits) or None if `cell_id` doesn't match
    the documented `<tech><root><zone>` shape."""
    m = CELL_ID_RE.match(cell_id)
    return m.groups() if m else None


def site_root(cell_id):
    """The physical-site part of a `cell_id` -- what actually has to change for
    the user to have *moved*. Falls back to the raw string if unparsable, so an
    unexpected id degrades to "its own site" instead of raising."""
    parsed = parse_cell_id(cell_id)
    return parsed[1] if parsed else cell_id


# ------------------------------------------------------------------ hour bands
# (name, start_hour, end_hour) with half-open bounds [start, end). A band whose
# start > end wraps midnight ("sleep" = 22h..07h). Four wide bands beat 5/6/8
# narrower ones on this dataset: every band keeps enough events to be estimated
# reliably, which raised the variance of per-user predictability explained by
# the group label from 0.65 (6 bands) to 0.69, and prefix-detectability from
# 85% to 86%.
DEFAULT_BANDS = [("sleep", 22, 7), ("morning", 7, 12), ("day", 12, 18), ("evening", 18, 22)]

DEFAULT_N_GROUPS = 4
DEFAULT_SHRINKAGE = 8.0


def band_of_hour(hour, bands=DEFAULT_BANDS):
    """Index of the band containing `hour`, honouring midnight-wrapping bands."""
    for i, (_name, start, end) in enumerate(bands):
        if start <= end:
            if start <= hour < end:
                return i
        elif hour >= start or hour < end:      # wraps midnight
            return i
    return 0


def band_transition_counts(cells, hours, bands=DEFAULT_BANDS):
    """(moves, transitions) per band for one user, counted on site-root changes.

    A "transition" is a consecutive pair of events; it is attributed to the band
    of the *later* event (the one being predicted). Returns two float arrays of
    length len(bands)."""
    n_bands = len(bands)
    moves = np.zeros(n_bands)
    totals = np.zeros(n_bands)
    if hours is None:
        return moves, totals
    roots = [site_root(c) for c in cells]
    for i in range(1, len(roots)):
        b = band_of_hour(hours[i], bands)
        totals[b] += 1
        moves[b] += roots[i] != roots[i - 1]
    return moves, totals


def fit_band_prior(users, bands=DEFAULT_BANDS):
    """Global per-band site-change rate over `users` -- the shrinkage target.
    Fit on the TRAIN split only, so test users never influence the geometry."""
    moves = np.zeros(len(bands))
    totals = np.zeros(len(bands))
    for _uid, cells, hours in users:
        m, t = band_transition_counts(cells, hours, bands)
        moves += m
        totals += t
    return moves / np.maximum(totals, 1.0)


def band_profile(cells, hours, prior, bands=DEFAULT_BANDS, shrinkage=DEFAULT_SHRINKAGE):
    """One user's per-band site-change rate, shrunk toward `prior`.

    `shrinkage` acts as that many pseudo-transitions already observed at the
    prior rate, so a band with few real transitions stays near the population
    rate instead of jumping to 0% or 100%.

    A band with no observed transitions *and* `shrinkage == 0` would otherwise
    be 0/0; it falls back to the prior rate for that band. Without this the
    profile would carry NaNs, every centroid distance would be NaN, and
    `GroupModel.assign` would silently put every user in group 0."""
    moves, totals = band_transition_counts(cells, hours, bands)
    numer = moves + shrinkage * prior
    denom = totals + shrinkage
    empty = denom == 0
    if empty.any():
        numer = np.where(empty, prior, numer)
        denom = np.where(empty, 1.0, denom)
    return numer / denom


# ---------------------------------------------------------------- group tokens
def group_token(group):
    """Dedicated tokenizer token for a behavioural group, e.g. '<G0>'."""
    return f"<G{group}>"


def group_vocab(n_groups=DEFAULT_N_GROUPS):
    return [group_token(g) for g in range(n_groups)]


# ------------------------------------------------------------------ the model
class GroupModel:
    """K-means over shrunk per-band site-change profiles, fitted on train users.

    Centroids are re-ordered by increasing mean site-change rate, so group 0 is
    always the most sedentary and group n-1 the most mobile whatever k-means'
    internal labelling was. Assignment is nearest centroid in Euclidean
    distance on the raw (un-standardised) profile: the bands are already
    commensurable rates in [0, 1], and standardising them made things worse --
    it inflates the sparse night band into the dominant axis (R^2 0.69 -> 0.43
    on this dataset)."""

    def __init__(self, centers, prior, bands, shrinkage, names=None):
        self.centers = np.asarray(centers, dtype=float)
        self.prior = np.asarray(prior, dtype=float)
        self.bands = list(bands)
        self.shrinkage = float(shrinkage)
        self.names = list(names) if names else [f"G{g}" for g in range(len(self.centers))]

    @property
    def n_groups(self):
        return len(self.centers)

    def profile(self, cells, hours):
        return band_profile(cells, hours, self.prior, self.bands, self.shrinkage)

    def assign(self, cells, hours):
        """Group of one user, from whatever slice of their day is passed in.

        Passing a *prefix* is what makes group detection legitimate at test
        time: the label comes only from events the model has already been
        given, never from the target it is being asked to predict."""
        if hours is None:
            return 0
        d = ((self.centers - self.profile(cells, hours)) ** 2).sum(axis=1)
        return int(np.argmin(d))

    def assign_all(self, users):
        return [self.assign(cells, hours) for _uid, cells, hours in users]


def fit_groups(users, n_groups=DEFAULT_N_GROUPS, bands=DEFAULT_BANDS,
               shrinkage=DEFAULT_SHRINKAGE, seed=42):
    """Fit a GroupModel on `users` (train split only). Requires scikit-learn."""
    from sklearn.cluster import KMeans

    prior = fit_band_prior(users, bands)
    X = np.array([band_profile(cells, hours, prior, bands, shrinkage)
                  for _uid, cells, hours in users])
    km = KMeans(n_clusters=n_groups, n_init=50, random_state=seed).fit(X)
    order = np.argsort(km.cluster_centers_.mean(axis=1))   # sedentary -> mobile
    return GroupModel(km.cluster_centers_[order], prior, bands, shrinkage)


# -------------------------------------------------- per-group loss weighting
def hour_persistence(users, labels, group, n_hours=24):
    """(same_cell_id, transitions) per hour for one group -- how often "repeat
    the previous cell_id" is correct at each hour. Uses raw `cell_id` equality,
    not site root: that is exactly the quantity the model's top-1 competes
    against."""
    same = np.zeros(n_hours)
    totals = np.zeros(n_hours)
    for (_uid, cells, hours), lab in zip(users, labels):
        if hours is None or lab != group:
            continue
        for i in range(1, len(cells)):
            h = hours[i] % n_hours
            totals[h] += 1
            same[h] += cells[i] == cells[i - 1]
    return same, totals


def derive_group_windows(users, labels, n_groups, min_support=25, min_window_events=60,
                        min_deficit=0.05, weight_gain=2.0, max_weight=4.0):
    """Per-group transition windows read off the training data.

    For each group, find the maximal runs of consecutive hours where that
    group's `cell_id` persistence sits at least `min_deficit` *below* the
    group's own average persistence -- the hours where the "repeat the last
    cell_id" baseline breaks down and the model actually has to reason.

    Hours with fewer than `min_support` observed transitions are ignored (too
    noisy to trust), and a candidate run needs `min_window_events` transitions
    in total to be kept, so a weight is never derived from a handful of events.

    The weight scales with how much harder the window is than the group's
    average::

        ratio  = group_mean_persistence / window_persistence
        weight = clip(1 + weight_gain * (ratio - 1), 1.0, max_weight)

    Returns {group: [(start_hour, end_hour, weight), ...]}. A group whose
    persistence is flat across the day legitimately gets an **empty** list: for
    G0 on this dataset "repeat the last cell_id" is uniformly strong (56%), so
    there is no hour worth upweighting and a flat weight of 1.0 is correct.
    """
    windows = {}
    for g in range(n_groups):
        same, totals = hour_persistence(users, labels, g)
        ok = totals >= min_support
        if not ok.any():
            windows[g] = []
            continue
        base = same[ok].sum() / totals[ok].sum()
        persistence = np.where(ok, same / np.maximum(totals, 1.0), np.inf)
        found = []
        h = 0
        while h < len(totals):
            if base - persistence[h] > min_deficit:
                start = h
                while h < len(totals) and base - persistence[h] > min_deficit:
                    h += 1
                n_events = totals[start:h].sum()
                if n_events >= min_window_events:
                    acc = same[start:h].sum() / n_events
                    ratio = base / max(acc, 1e-6)
                    w = float(np.clip(1.0 + weight_gain * (ratio - 1.0), 1.0, max_weight))
                    found.append((int(start), int(h), round(w, 2)))
            else:
                h += 1
        windows[g] = found
    return windows


def windows_for(group, group_windows, fallback=()):  # noqa: D401
    """Windows of `group`, or `fallback` when the group has none configured."""
    return group_windows.get(group, list(fallback))


# --------------------------------------------------------------- description
def describe_groups(users, labels, model, n_groups):
    """Per-group summary rows for printing: size, band profile, persistence."""
    rows = []
    counts = Counter(labels)
    for g in range(n_groups):
        idx = [i for i, lab in enumerate(labels) if lab == g]
        if not idx:
            rows.append({"group": g, "n": 0})
            continue
        profiles = np.array([model.profile(users[i][1], users[i][2]) for i in idx])
        pers, lens = [], []
        for i in idx:
            cells = users[i][1]
            lens.append(len(cells))
            if len(cells) > 1:
                pers.append(np.mean([cells[j] == cells[j - 1] for j in range(1, len(cells))]))
        rows.append({
            "group": g,
            "n": counts[g],
            "share": counts[g] / len(labels) if labels else 0.0,
            "profile": profiles.mean(axis=0),
            "persistence": float(np.mean(pers)) if pers else float("nan"),
            "mean_len": float(np.mean(lens)),
        })
    return rows
''',
    'generate_sample_users': r'''#!/usr/bin/env python3
"""Génère des fichiers <N>_users_{train,test}.jsonl synthétiques, au même format
que le jeu réel produit par `make data-train`.

Sert de **repli autonome** : le notebook peut tourner de bout en bout sans aucun
fichier de données externe. Les séquences portent donc, comme les données réelles :

  * des `cell_id` au format documenté dans CLAUDE.md -- `<lettre techno><site
    root><chiffres de zone>` -- de sorte que plusieurs `cell_id` partagent le même
    site physique (c'est ce que `user_groups.site_root()` compare) ;
  * une **heure** par événement (champ `hours`), sans laquelle le cross-training
    par groupe de comportement se désactive faute de signal horaire ;
  * quatre **archétypes de mobilité** calqués sur les groupes observés dans les
    vraies données, pour que le regroupement ait quelque chose à retrouver :
    sédentaire / régulier / actif l'après-midi / actif le matin.

Usage :
    python generate_sample_users.py --n-users 500 [--seed 42] [--out-dir DIR]
    python generate_sample_users.py --n-train 400 --n-test 100 --out-dir data/dataset_for_training
"""
import argparse
import json
import random
from pathlib import Path

SYSTEM_PROMPT = ("You are an AI that generates a user's cell-visit trajectory. "
                 "Output format: User <USER_ID> | <N_RECORDS> events | "
                 "<CELL_ID> <CELL_ID> ...")
USER_PROMPT = "Here is the sequence for this user."

# Archétypes : (nom, probabilité de changer de site par tranche, événements/heure).
# Les tranches sont (nuit 22-07, matin 07-12, jour 12-18, soir 18-22), dans l'ordre
# de CONFIG["group_bands"]. Les valeurs reprennent les profils mesurés sur les 400
# vrais utilisateurs (cf. python/user_groups.py) -- y compris le fait que les
# sédentaires émettent nettement plus d'événements par heure.
ARCHETYPES = [
    ("sedentaire",     (0.28, 0.32, 0.34, 0.38), 10.0, 0.30),
    ("regulier",       (0.31, 0.47, 0.46, 0.45),  9.0, 0.22),
    ("actif_apresmidi",(0.30, 0.47, 0.67, 0.60),  4.6, 0.15),
    ("actif_matin",    (0.35, 0.67, 0.65, 0.43),  4.8, 0.15),
]
BANDS = ((22, 7), (7, 12), (12, 18), (18, 22))


def band_of_hour(hour):
    for i, (start, end) in enumerate(BANDS):
        if start <= end:
            if start <= hour < end:
                return i
        elif hour >= start or hour < end:
            return i
    return 0


def load_real_vocab():
    """Vocabulaire repris des fichiers réels s'ils sont là, sinon []."""
    vocab = set()
    for path in ("100_users_train.jsonl", "100_users_test.jsonl"):
        try:
            with open(path, encoding="utf-8") as f:
                for line in f:
                    row = json.loads(line)
                    if "cells" in row:
                        vocab.update(row["cells"])
                    else:
                        content = row["conversations"][-1]["content"]
                        vocab.update(content.split("|", 2)[2].split())
        except (FileNotFoundError, KeyError, ValueError):
            pass
    return sorted(vocab)


def synthetic_vocab(n_sites=110):
    """Vocabulaire au format CLAUDE.md : pour chaque site, des cellules 2G
    (`B<root><1-4>`) et 3G (`U<root><1|2><01-04>`). Plusieurs `cell_id` par site
    physique, exactement comme dans les vraies données."""
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    sites = {}
    for i in range(n_sites):
        root = "KV" + letters[i // 26 % 26] + letters[i % 26] + letters[(i * 7) % 26]
        cells = [f"B{root}{s}" for s in range(1, 5)]
        cells += [f"U{root}{z}{s:02d}" for z in (1, 2) for s in range(1, 5)]
        sites[root] = cells
    return sites


def sites_from_vocab(vocab):
    """Regroupe un vocabulaire plat par site physique (préfixe sans techno/zone)."""
    import re
    pat = re.compile(r"^([BDUV])([A-Z]+?)((?:[12]\d{2})|\d)$")
    sites = {}
    for cell in vocab:
        m = pat.match(cell)
        root = m.group(2) if m else cell
        sites.setdefault(root, []).append(cell)
    return sites


def make_sequence(rng, sites, archetype):
    """Une journée : répertoire de sites restreint, mobilité dépendant de l'heure.

    Un « déplacement » change de site ; sinon on reste sur place, en réémettant
    parfois une AUTRE cellule du même site (changement techno/secteur) -- c'est
    ce bruit qui rend `site_root()` indispensable côté analyse."""
    _name, move_probs, per_hour, same_site_switch = archetype
    roots = rng.sample(sorted(sites), k=rng.randint(12, 24))
    home = roots[0]
    start = rng.choices([0, 1, 6, 7, 8, 9, 10], weights=[15, 10, 12, 18, 16, 15, 14])[0]
    end = rng.choices([15, 16, 17, 18, 19, 22, 23], weights=[12, 14, 16, 12, 12, 16, 18])[0]
    if end <= start:
        end = min(23, start + 6)

    cells, hours = [], []
    current = home
    current_cell = rng.choice(sites[home])
    for hour in range(start, end + 1):
        n_events = max(1, int(rng.gauss(per_hour, per_hour * 0.35)))
        p_move = move_probs[band_of_hour(hour)]
        for _ in range(n_events):
            if rng.random() < p_move:
                # déplacement : nouveau site, donc nouvelle cellule
                current = rng.choice([r for r in roots if r != current])
                current_cell = rng.choice(sites[current])
            elif rng.random() < same_site_switch:
                # sur place, mais le téléphone se raccroche à une AUTRE cellule du
                # même site (changement techno/secteur) : le cell_id change sans
                # déplacement -- c'est ce bruit qui rend site_root() indispensable
                others = [c for c in sites[current] if c != current_cell]
                if others:
                    current_cell = rng.choice(others)
            # sinon : on ne bouge pas et on réémet EXACTEMENT la même cellule,
            # ce qui fait de « recopier le cell_id précédent » une baseline forte
            cells.append(current_cell)
            hours.append(hour)
    # borne de longueur, comme dans les vraies données (19 à 200 événements)
    if len(cells) > 200:
        cells, hours = cells[:200], hours[:200]
    while len(cells) < 19:
        cells.append(cells[-1] if cells else rng.choice(sites[home]))
        hours.append(hours[-1] if hours else start)
    return cells, hours


def make_line(user_id, cells, hours, with_hours=True):
    """Même schéma que python/format_for_train.py : champs structurés
    `cells`/`hours` (ce que le notebook lit) + le texte `conversations`."""
    row = {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT},
            {"role": "assistant",
             "content": f"User {user_id} | {len(cells)} events | {' '.join(cells)}"},
        ],
        "user_id": str(user_id),
        "cells": cells,
    }
    if with_hours:
        row["hours"] = hours
    return json.dumps(row, ensure_ascii=False)


def generate(n_train, n_test, seed, out_dir, with_hours=True, stem=None):
    rng = random.Random(seed)
    vocab = load_real_vocab()
    if vocab:
        sites = sites_from_vocab(vocab)
    else:
        sites = synthetic_vocab()
        print("⚠️  Fichiers 100_users introuvables : vocabulaire synthétique utilisé "
              f"({len(sites)} sites au format CLAUDE.md).")

    n_total = n_train + n_test
    user_ids = rng.sample(range(19_000_000, 21_000_000), n_total)
    lines = []
    for i, uid in enumerate(user_ids):
        arch = ARCHETYPES[i % len(ARCHETYPES)]     # les 4 archétypes équirépartis
        cells, hours = make_sequence(rng, sites, arch)
        lines.append(make_line(uid, cells, hours, with_hours))
    rng.shuffle(lines)

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = stem or f"{n_train}_users"
    written = []
    for name, chunk in ((f"{stem}_train.jsonl", lines[:n_train]),
                        (f"{stem}_test.jsonl", lines[n_train:])):
        dest = out_dir / name
        dest.write_text("\n".join(chunk) + "\n", encoding="utf-8")
        print(f"{dest} : {len(chunk)} utilisateurs")
        written.append(dest)
    return written


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--n-users", type=int, default=None,
                        help="nombre total d'utilisateurs (split 80/20 train/test)")
    parser.add_argument("--n-train", type=int, default=None, help="nombre exact de lignes train")
    parser.add_argument("--n-test", type=int, default=None, help="nombre exact de lignes test")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--out-dir", type=Path, default=Path("."),
                        help="dossier de sortie (défaut : dossier courant)")
    parser.add_argument("--stem", default=None,
                        help="préfixe des fichiers (défaut : <n_train>_users)")
    parser.add_argument("--no-hours", action="store_true",
                        help="ne pas émettre le champ hours (désactive le mode heure du notebook)")
    args = parser.parse_args()

    if args.n_train is None or args.n_test is None:
        total = args.n_users if args.n_users is not None else 500
        n_train = int(total * 0.8)
        n_test = total - n_train
    else:
        n_train, n_test = args.n_train, args.n_test

    generate(n_train, n_test, args.seed, args.out_dir,
             with_hours=not args.no_hours, stem=args.stem)


if __name__ == "__main__":
    main()
''',
}
# --- FIN DU BLOC GÉNÉRÉ ---

# Écriture des modules embarqués UNIQUEMENT s'ils ne sont pas importables : sur un
# dépôt normalement installé, python/ est dans sys.path et rien n'est écrit.
_written = []
for _name, _code in _EMBEDDED_MODULES.items():
    try:
        __import__(_name)
    except ImportError:
        Path(f"{_name}.py").write_text(_code, encoding="utf-8")
        _written.append(_name)
if _written:
    import importlib
    importlib.invalidate_caches()
    print("Modules non trouvés — générés automatiquement dans le répertoire courant ✅ : "
          + ", ".join(f"{m}.py" for m in _written))


# ============ HYPERPARAMÈTRES — IDENTIQUES POUR LES 7 JOURS ============
# Toute valeur modifiée ici s'applique aux 7 entraînements : c'est la condition
# pour que la comparaison entre jours mesure la donnée, et pas le réglage.
CONFIG = {
    # Modèle de base — décommentez celui voulu :
    # "model_name": "mistralai/Mistral-7B-v0.1",      # 7B : GPU obligatoire, 4-bit (QLoRA) ; compter ~8x le temps
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",       # 0.5B : le bon choix pour 7 runs comparatifs

    "output_dir": "outputs_week",
    "seed": 42,

    # Token Hugging Face (optionnel : Qwen2.5 est public). Ordre de priorité au login :
    # variable d'env HF_TOKEN > secret Colab "HF_TOKEN" > la valeur ci-dessous.
    # ⚠️ Si vous partagez ce notebook, videz cette valeur et utilisez les secrets Colab.
    "hf_token": "",

    # --- Arrêt de l'entraînement ---
    # IDENTIQUES au notebook d'origine, et il ne faut PAS y toucher pour gagner du
    # temps : `max_epochs` est l'horizon du scheduler cosine (voir plus bas), donc
    # le baisser ne raccourcit pas seulement l'entraînement, il fait décroître le
    # learning rate beaucoup plus vite. Un jour entraîné avec max_epochs=12 n'est
    # comparable ni aux runs précédents, ni à un jour entraîné avec 20.
    # Le pic arrive vers l'epoch 9 et la patience coupe vers 13 : le plafond de 20
    # ne coûte donc presque rien en pratique.
    "early_stop_patience": 4,
    "max_epochs": 20,

    # --- LoRA / optimisation (réglés anti-overfit : dataset très petit) ---
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "learning_rate": 1.5e-4,
    "weight_decay": 0.01,
    "max_seq_len": 512,          # longueur max à l'évaluation (séquence complète)
    "chunk_len": 128,            # fenêtres d'entraînement courtes -> mémoire réduite (en nb d'événements)
    "chunk_stride": 64,
    # Fraction de la séquence de chaque utilisateur donnée en contexte : le modèle
    # n'est entraîné QUE sur ce préfixe, et doit prédire la suite (1 - context_fraction)
    # jamais vue à l'entraînement. Appliqué à TOUS les utilisateurs (train ET test).
    "context_fraction": 0.8,
    "prefix_text": "Events:",

    # Poids d'échantillonnage pour le WeightedRandomSampler : les fenêtres (chunks)
    # d'entraînement contenant au moins une transition en créneau sont tirées avec
    # ce poids (vs 1.0 par défaut).
    "transition_chunk_oversample": 3.0,

    # Créneau GLOBAL de repli. Utilisé uniquement quand les groupes sont désactivés
    # (use_groups=False) ou quand le jeu n'a pas d'heures.
    "transition_windows": [(4, 6, 2.0), (18, 20, 4.0)],

    # ============ CROSS-TRAINING PAR GROUPE DE COMPORTEMENT ============
    # Un groupe = un profil de « taux de changement de site physique par tranche
    # horaire ». Les centroïdes sont réajustés SUR LE TRAIN DE CHAQUE JOUR (un
    # groupe doit rester dérivable des seules données du jour), donc les effectifs
    # par groupe varient d'un jour à l'autre. Les groupes sont numérotés par
    # mobilité croissante, ce qui garde G0 = « le plus sédentaire » d'un jour sur
    # l'autre — mais ne comparez pas les parts au point de pourcentage près.
    "use_groups": True,
    "n_groups": 4,
    "group_bands": [("sleep", 22, 7), ("morning", 7, 12), ("day", 12, 18), ("evening", 18, 22)],
    "group_shrinkage": 8.0,
    "group_windows": None,              # None => dérivés des données d'entraînement du jour
    "group_window_min_deficit": 0.05,   # écart de persistance minimal pour retenir une heure
    "group_window_weight_gain": 2.0,    # poids = 1 + gain x (persist_moyenne/persist_creneau - 1)
    "group_window_max_weight": 4.0,     # plafond de poids
    "group_detect_fraction": 0.5,       # préfixe de détection quand context_fraction=None

    # --- Guards (protection machine) ---
    "min_free_ram_gb": 2.0,
    "max_ram_used_fraction": 0.90,
    "min_free_disk_gb": 5.0,
    "ram_check_every_steps": 10,
    "save_total_limit": 1,
}


# ============ PILOTAGE DE LA SEMAINE ============
WEEK_CONFIG = {
    # Les 7 jours, dans l'ordre d'exécution.
    "days": ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"],
    # Marqueur de semaine présent dans le nom de fichier ("w1"), ou None si absent.
    "week": "w1",

    # Où chercher les 14 fichiers JSONL. Chaque dossier est exploré à plat ET sur un
    # niveau de sous-dossier. Ajoutez le vôtre en tête si besoin.
    "data_dirs": [
        "/content",
        "/content/data",
        "/content/drive/MyDrive/cellid_week",
        "data/data_nino_15days",
        "data",
        ".",
    ],

    # Résultats : un JSON par jour, écrits au fur et à mesure. Sur Colab avec
    # use_drive=True, ils atterrissent sur Drive et SURVIVENT à une déconnexion.
    "use_drive": True,
    "drive_results_dir": "/content/drive/MyDrive/cellid_week/results",
    "local_results_dir": "outputs_week/results",

    # Jours à refaire même si leur JSON existe déjà (ex. ["monday"]). [] = aucun.
    "force_rerun": [],

    # --- Vitesse ---
    # Répartition mesurée du coût d'une epoch sur ce jeu (1900 utilisateurs) :
    # 73 % entraînement, 27 % validation. Tout le bloc d'évaluation finale
    # (fractions, groupes, ablations) ne pèse que ~2 % de la journée : le couper
    # ne rapporte rien, et il porte l'essentiel de la matière de présentation.
    #
    # Validation par epoch : sous-échantillon FIXE (tiré au seed, donc le même
    # d'un jour à l'autre) des utilisateurs du train. 400 fait tomber la
    # validation de 27 % à 7 % d'une epoch, soit ~20 % de temps gagné sur la
    # journée. Contrepartie : la métrique d'early stopping repose sur ~6k
    # prédictions au lieu de ~30k, donc l'epoch retenue peut se décaler d'une.
    # Mettre None pour valider sur TOUS les utilisateurs du train, comme le
    # notebook d'origine — c'est le réglage exact, pas le réglage rapide.
    "val_subsample": 400,
    # Taille de batch de l'évaluation. 16 est un réglage GRATUIT : le score est
    # inchangé (padding à droite + attention_mask) et des batches de 8 séquences
    # de ~155 tokens sous-utilisent le GPU. 32 sur L4/A100, 16 sur T4.
    # À baisser si OOM pendant une éval : le pic mémoire est
    # batch × longueur × taille_du_vocabulaire_complet pour les logits
    # (32 × 512 × 152k en bf16 ≈ 5 Go, ce qui est juste sur un T4 16 Go).
    "eval_batch_size": 16,
    # Batch d'entraînement. None => valeur automatique selon le device.
    "train_batch_size": None,
    "grad_accum": None,

    # --- Ce qu'on mesure à la fin de chaque jour ---
    # Fractions de contexte évaluées sur le test. CONFIG["context_fraction"] est
    # toujours ajoutée en tête : c'est le protocole de référence.
    "test_context_fractions": [0.3, 0.5, 0.9],
    "run_global_eval": True,     # évaluation sur 100% de l'historique (diagnostic)
    "run_group_ablation": True,  # détecté / oracle / sans token / groupe forcé
    "run_hour_ablation": True,   # heures réelles / mélangées / fixes / décalées
    "save_adapters": True,       # adaptateur LoRA du meilleur modèle, par jour

    # Rognage du nombre d'utilisateurs par jour. Deux usages :
    #   - tour de chauffe (ex. 60) : valide les 7 jours en ~2 min. Sans risque,
    #     la boucle refait un jour dont le JSON porte un autre limit_users.
    #   - dernier recours de vitesse (ex. 950) : entraînement ET validation
    #     baissent ensemble, soit ~50 % de temps en moins. Les 950 premiers
    #     utilisateurs sont représentatifs (longueur moyenne 76,0 contre 77,2
    #     sur les 1900 ; médiane 62 contre 61), et appliqué aux 7 jours cela
    #     garde la comparaison jour à jour valide — mais les scores absolus
    #     baissent et ne sont plus comparables aux runs précédents.
    # None en usage normal.
    "limit_users": None,
}

print(f"Modèle : {CONFIG['model_name']}")
print(f"Semaine : {' → '.join(WEEK_CONFIG['days'])} ({WEEK_CONFIG['week'] or 'sans marqueur de semaine'})")
print(f"Plafond : {CONFIG['max_epochs']} epochs, arrêt après {CONFIG['early_stop_patience']} sans progrès")

In [ ]:
# ==================== DÉTECTION ENVIRONNEMENT (local / Colab / Drive) ====================
import os, subprocess, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Google Colab détecté — installation des dépendances…")
    # torchao préinstallé sur Colab (0.10) est incompatible avec peft récent -> on le retire
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.45", "peft>=0.15", "datasets>=3.0",
                    "accelerate>=1.0", "psutil", "bitsandbytes"], check=True)

# --- Drive monté MAINTENANT, pas au moment d'écrire ---
# L'authentification Drive est interactive : la déclencher au bout de trois heures
# d'entraînement, c'est la rater et perdre les résultats du jour.
RESULTS_DIR = Path(WEEK_CONFIG["local_results_dir"])
if IN_COLAB and WEEK_CONFIG["use_drive"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS_DIR = Path(WEEK_CONFIG["drive_results_dir"])
        print(f"Drive monté — les résultats survivront à une déconnexion.")
    except Exception as exc:
        print(f"⚠️  Drive non monté ({exc}) — repli sur {RESULTS_DIR} (perdu si la session tombe).")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Résultats : {RESULTS_DIR.resolve()}")

# --- Authentification Hugging Face (facultative, le modèle est public) ---
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if IN_COLAB and not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or CONFIG["hf_token"]
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Authentifié sur Hugging Face Hub")

import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print(f"GPU : {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} Go)")
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float32   # fp32 : le plus stable sur Apple Silicon
else:
    DEVICE, DTYPE = "cpu", torch.float32

# Guard : ne pas saturer les cœurs CPU en local (machine utilisable pendant l'entraînement)
if DEVICE != "cuda":
    torch.set_num_threads(max(1, (os.cpu_count() or 4) // 2))

print(f"Environnement : {'Google Colab' if IN_COLAB else 'local'} | device={DEVICE} | dtype={DTYPE}")
if DEVICE != "cuda":
    print("⚠️  Sept entraînements hors GPU, ce n'est pas réaliste : lancez ce notebook sur Colab "
          "avec un GPU (Exécution > Modifier le type d'exécution).")

In [ ]:
# ==================== PRÉFLIGHT : RESSOURCES + LES 14 FICHIERS ====================
# Tout ce qui peut échouer doit échouer ICI, avant la première seconde de GPU :
# un fichier manquant découvert au jour 5 coûte les quatre jours déjà calculés.
import shutil, psutil, json

def _search_dirs():
    seen, dirs = set(), []
    for d in WEEK_CONFIG["data_dirs"]:
        p = Path(d)
        if p.is_dir() and p.resolve() not in seen:
            seen.add(p.resolve())
            dirs.append(p)
    return dirs

def find_day_file(day, split):
    """Le JSONL de (jour, split). `split` vaut "train" ou "test".
    Un nom doit contenir le jour, le marqueur de semaine, et train/test —
    "..._users_training.jsonl" contient "train", "..._users_test.jsonl" ne le
    contient pas, donc les deux motifs ne se recouvrent jamais."""
    week = (WEEK_CONFIG["week"] or "").lower()
    found = {}
    for d in _search_dirs():
        for path in list(d.glob("*.jsonl")) + list(d.glob("*/*.jsonl")):
            name = path.name.lower()
            if day not in name or split not in name:
                continue
            if week and week not in name:
                continue
            if split == "train" and "test" in name:
                continue
            found[path.resolve()] = path
    paths = sorted(found.values(), key=lambda p: str(p))
    if not paths:
        raise FileNotFoundError(
            f"Aucun fichier pour {day}/{split}. Cherché un nom contenant "
            f"'{day}' + '{split}'" + (f" + '{week}'" if week else "") + " dans : "
            + ", ".join(str(d) for d in _search_dirs())
            + ". Ajoutez le bon dossier à WEEK_CONFIG['data_dirs'], ou mettez "
              "WEEK_CONFIG['week']=None si vos noms ne portent pas de marqueur de semaine.")
    if len(paths) > 1:
        raise ValueError(
            f"{len(paths)} fichiers correspondent à {day}/{split} : "
            + ", ".join(str(p) for p in paths)
            + ". Affinez WEEK_CONFIG['data_dirs'] ou WEEK_CONFIG['week'] pour lever l'ambiguïté.")
    return paths[0]

def preflight():
    vm = psutil.virtual_memory()
    free_ram_gb = vm.available / 1e9
    free_disk_gb = shutil.disk_usage(".").free / 1e9
    print(f"RAM libre : {free_ram_gb:.1f} Go ({vm.percent:.0f}% utilisée) | "
          f"Disque libre : {free_disk_gb:.1f} Go")
    if free_ram_gb < CONFIG["min_free_ram_gb"]:
        raise RuntimeError(f"RAM libre insuffisante (< {CONFIG['min_free_ram_gb']} Go).")
    if free_disk_gb < CONFIG["min_free_disk_gb"]:
        raise RuntimeError(f"Espace disque insuffisant (< {CONFIG['min_free_disk_gb']} Go).")

    # Le cross-training par groupe a besoin de scikit-learn (user_groups lui-même est
    # embarqué dans la cellule CONFIG). On échoue ICI avec un message clair.
    if CONFIG.get("use_groups"):
        try:
            import user_groups            # noqa: F401
            import sklearn.cluster        # noqa: F401
        except ImportError as exc:
            raise ImportError(
                f"CONFIG['use_groups'] est True mais l'import a échoué ({exc}). "
                "Installez scikit-learn, ou mettez use_groups=False.") from exc

    files = {}
    print(f"\nFichiers ({len(WEEK_CONFIG['days'])} jours × 2) :")
    for day in WEEK_CONFIG["days"]:
        files[day] = {s: find_day_file(day, s) for s in ("train", "test")}
        tr, te = files[day]["train"], files[day]["test"]
        print(f"  {day:<10} train {tr.name:<40} ({tr.stat().st_size / 1e6:.1f} Mo)")
        print(f"  {'':<10} test  {te.name:<40} ({te.stat().st_size / 1e6:.1f} Mo)")
    print("\nPréflight OK ✅")
    return files

DAY_FILES = preflight()

In [ ]:
# ==================== CHARGEMENT DES 14 FICHIERS (UNE SEULE FOIS) ====================
# Les 14 fichiers sont lus maintenant, intégralement, et gardés en RAM pour les 7
# entraînements. Trois raisons :
#   1. Le vocabulaire cell_id partagé doit être connu AVANT d'étendre le tokenizer,
#      qui n'est construit qu'une fois.
#   2. Un fichier corrompu doit faire échouer le notebook avant le premier GPU.
#   3. Les baselines par jour tombent immédiatement, sans une seconde de calcul —
#      c'est déjà de la matière de comparaison, et la référence de tout le reste.
# Coût : ~30 Mo de RAM une fois les cell_id internés (369 chaînes distinctes
# partagées entre 1,1 million d'événements, au lieu d'autant d'objets).
from collections import Counter
from cellid_encoding import split_index

_INTERN = {}

def _intern(cell):
    got = _INTERN.get(cell)
    if got is None:
        got = _INTERN[cell] = sys.intern(cell)
    return got

def load_users(path, limit=None):
    """Parse le JSONL -> [(user_id, [cell_id, ...], [heure, ...] | None), ...].
    Si une ligne porte les champs structurés "cells"/"hours" (voir
    python/format_for_train.py), ils sont utilisés directement. Sinon on retombe
    sur le texte "assistant" -- heures = None -> pas d'heure interleavée."""
    users = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if limit is not None and len(users) >= limit:
                break
            row = json.loads(line)
            if "cells" in row:
                uid = str(row.get("user_id", "?"))
                cells = [_intern(c) for c in row["cells"]]
                hours = row.get("hours")
            else:
                content = row["conversations"][-1]["content"]
                head, _, seq = content.split("|", 2)
                uid = head.split()[1]
                cells = [_intern(c) for c in seq.split()]
                hours = None
            users.append((uid, cells, hours))
    return users

def split_index_for(cells, context_fraction):
    return split_index(len(cells), context_fraction) if context_fraction is not None else 1

def baseline_persistence(users, context_fraction=None):
    """Prédire « même cell_id que le précédent » — la référence à battre.
    Si context_fraction est fourni, ne compte que les positions de la cible, pour
    rester comparable à evaluate_cell_accuracy(..., context_fraction=...)."""
    tot = ok = 0
    for _, s, _ in users:
        min_j = split_index_for(s, context_fraction)
        for i in range(1, len(s)):
            if i < min_j:
                continue
            tot += 1
            ok += s[i] == s[i - 1]
    return ok / tot if tot else float("nan")

def baseline_persistence_breakdown(users, context_fraction, transition_windows):
    """Comme baseline_persistence, mais rapporte l'accuracy séparément dans / hors
    créneaux de transition. `transition_windows` : liste globale (debut, fin, poids)
    OU callable user_index -> liste de créneaux (cas des créneaux par groupe)."""
    win_of = transition_windows if callable(transition_windows) else (lambda _i: transition_windows)
    all_wins = sorted({(s, e) for i in range(len(users)) for s, e, _ in win_of(i)})
    tot = ok = 0
    tot_in = ok_in = tot_out = ok_out = 0
    win_hits = {w: 0 for w in all_wins}
    win_tots = {w: 0 for w in all_wins}
    for ui, (_, s, hours) in enumerate(users):
        wins = win_of(ui)
        min_j = split_index_for(s, context_fraction)
        for i in range(1, len(s)):
            if i < min_j:
                continue
            tot += 1
            hit = s[i] == s[i - 1]
            ok += hit
            if hours is not None:
                h = hours[i]
                in_any = False
                for (start, end, _) in wins:
                    if start <= h < end:
                        win_tots[(start, end)] += 1
                        win_hits[(start, end)] += hit
                        in_any = True
                        break
                if in_any:
                    tot_in += 1; ok_in += hit
                else:
                    tot_out += 1; ok_out += hit
    res = {"overall": ok / tot if tot else float("nan"),
           "in_window": ok_in / tot_in if tot_in else float("nan"),
           "out_window": ok_out / tot_out if tot_out else float("nan"),
           "window_breakdown": {}}
    for w in all_wins:
        wtot = win_tots[w]
        res["window_breakdown"][w] = {"acc": win_hits[w] / wtot if wtot else float("nan"),
                                      "n": wtot}
    return res

# --- lecture -----------------------------------------------------------------
LIMIT = WEEK_CONFIG["limit_users"]
DAY_DATA = {}
for day in WEEK_CONFIG["days"]:
    DAY_DATA[day] = {
        "train": load_users(DAY_FILES[day]["train"], LIMIT),
        "test": load_users(DAY_FILES[day]["test"], None if LIMIT is None else max(4, LIMIT // 4)),
    }
if LIMIT is not None:
    print(f"⚠️  MODE TEST : {LIMIT} utilisateurs max par jour — résultats non représentatifs.\n")

# --- vocabulaire PARTAGÉ : l'union des 7 jours -------------------------------
# C'est ce qui rend les top-k comparables d'un jour à l'autre : sinon un jour au
# répertoire plus large affronte un softmax plus difficile, et l'écart mesuré n'est
# plus celui qu'on croit.
VOCAB = sorted({c for d in DAY_DATA.values()
                for split in ("train", "test")
                for _, cells, _ in d[split] for c in cells})
HAS_HOURS = any(hours is not None
                for d in DAY_DATA.values() for split in ("train", "test")
                for _, _, hours in d[split])

print(f"Vocabulaire PARTAGÉ (union des {len(WEEK_CONFIG['days'])} jours) : {len(VOCAB)} cell_id")
print("Heures disponibles : "
      + ("oui" if HAS_HOURS else "NON — tokens heure, groupes et pondération désactivés"))
if not HAS_HOURS:
    print("   ⚠️  Vos JSONL ne portent pas de timestamps : le modèle entraîné sera la variante")
    print("       amputée (pas de <Hxx>, pas de <Gk>). Régénérez-les avec python/format_for_train.py")
    print("       depuis les CSV sources si vous voulez le pipeline complet.")

# --- tableau par jour : la comparaison commence ici, avant tout GPU ----------
REF_FRAC = CONFIG["context_fraction"]
DAY_STATS = {}
print(f"\n{'jour':<11} {'train':>6} {'test':>5} {'événem.':>9} {'moy.':>6} {'vocab':>6} "
      f"{'baseline test':>14}")
for day in WEEK_CONFIG["days"]:
    tr, te = DAY_DATA[day]["train"], DAY_DATA[day]["test"]
    lens = [len(c) for _, c, _ in tr + te]
    vocab_day = {c for _, cells, _ in tr + te for c in cells}
    base = baseline_persistence(te, REF_FRAC)
    DAY_STATS[day] = {"n_train": len(tr), "n_test": len(te), "n_events": sum(lens),
                      "mean_len": sum(lens) / len(lens), "vocab": len(vocab_day),
                      "baseline_test": base,
                      "baseline_train": baseline_persistence(tr, REF_FRAC)}
    s = DAY_STATS[day]
    print(f"{day:<11} {s['n_train']:>6} {s['n_test']:>5} {s['n_events']:>9} "
          f"{s['mean_len']:>6.0f} {s['vocab']:>6} {base:>13.1%}")
print(f"\nBaseline = « répéter le dernier cell_id », sur la cible ({1 - REF_FRAC:.0%} finale)")
print("de chaque utilisateur de test. C'est la difficulté propre à l'échantillon du jour :")
print("les 7 jeux n'ayant AUCUN utilisateur en commun, c'est l'écart à cette ligne — pas")
print("l'accuracy brute — qui se compare d'un jour à l'autre.")

In [ ]:
# ==================== TOKENIZER PARTAGÉ + FABRIQUE DE MODÈLES ====================
# Le tokenizer est construit UNE fois. Les 7 jours partagent donc exactement les
# mêmes ids de tokens, la même taille d'embedding et la même taille de softmax.
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model
from cellid_encoding import HOUR_VOCAB
from user_groups import group_vocab

set_seed(CONFIG["seed"])

# Un modèle >= 7B ne tient pas en local : GPU obligatoire, chargé en 4-bit (QLoRA)
BIG_MODEL = any(t in CONFIG["model_name"] for t in ("7B", "8B", "13B", "70B"))
if BIG_MODEL and DEVICE != "cuda":
    raise RuntimeError(f"{CONFIG['model_name']} nécessite un GPU (Colab).")

USE_GROUPS = bool(CONFIG.get("use_groups")) and HAS_HOURS
if bool(CONFIG.get("use_groups")) and not HAS_HOURS:
    print("⚠️  use_groups=True mais le jeu n'a pas d'heures → cross-training par groupe "
          "désactivé (les groupes sont définis par tranche horaire).")
N_GROUPS = CONFIG["n_groups"] if USE_GROUPS else 0
GROUP_VOCAB = group_vocab(N_GROUPS) if USE_GROUPS else []

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:          # Mistral n'a pas de pad token
    tokenizer.pad_token = tokenizer.eos_token
n_added = tokenizer.add_tokens(VOCAB)   # 1 cell_id = 1 token dédié
print(f"{n_added} tokens cell_id ajoutés au tokenizer (vocabulaire de la semaine)")
if HAS_HOURS:
    print(f"{tokenizer.add_tokens(HOUR_VOCAB)} tokens heure ajoutés")
if USE_GROUPS:
    print(f"{tokenizer.add_tokens(GROUP_VOCAB)} tokens de groupe ajoutés ({', '.join(GROUP_VOCAB)})")

CELL_TOKEN_IDS = tokenizer.convert_tokens_to_ids(VOCAB)
HOUR_TOKEN_IDS = tokenizer.convert_tokens_to_ids(HOUR_VOCAB) if HAS_HOURS else []
GROUP_TOKEN_IDS = tokenizer.convert_tokens_to_ids(GROUP_VOCAB) if USE_GROUPS else []
NEW_TOKEN_IDS = CELL_TOKEN_IDS + HOUR_TOKEN_IDS + GROUP_TOKEN_IDS

PREFIX_IDS = tokenizer(CONFIG["prefix_text"], add_special_tokens=False).input_ids
CELL_TO_ID = {c: i for c, i in zip(VOCAB, CELL_TOKEN_IDS)}
HOUR_TO_ID = {h: i for h, i in zip(HOUR_VOCAB, HOUR_TOKEN_IDS)} if HAS_HOURS else {}
GROUP_TO_ID = {g: i for g, i in zip(GROUP_VOCAB, GROUP_TOKEN_IDS)} if USE_GROUPS else {}

CELL_IDS_T = torch.tensor(CELL_TOKEN_IDS)
CELL_POS = {tid: i for i, tid in enumerate(CELL_TOKEN_IDS)}   # id de token -> indice dans VOCAB


def new_model():
    """Un modèle de base NEUF, enveloppé dans un adaptateur LoRA NEUF.

    Recharger la base à chaque jour plutôt que de désenvelopper l'adaptateur du
    jour précédent : `get_peft_model` modifie le modèle EN PLACE (il injecte les
    couches LoRA dans les modules ciblés), donc le rappeler sur un modèle déjà
    enveloppé empile les adaptateurs et fait démarrer le jour N sur les poids du
    jour N-1. Le rechargement vient du cache HF local (quelques secondes pour un
    0.5B) : c'est le prix d'une garantie d'isolation, pas d'un vrai coût.

    `trainable_token_indices` fait vivre les embeddings des nouveaux tokens DANS
    l'adaptateur PEFT : ils repartent donc eux aussi de zéro à chaque jour.
    """
    set_seed(CONFIG["seed"])   # même init LoRA pour les 7 jours : l'écart mesuré
                               # vient des données, pas du tirage d'initialisation
    if BIG_MODEL:
        from transformers import BitsAndBytesConfig
        from peft import prepare_model_for_kbit_training
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE)
        model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"],
                                                     quantization_config=bnb, device_map={"": 0})
    else:
        model = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"], dtype=DTYPE)

    # Redimensionnement AVANT prepare_model_for_kbit_training, init simple (mean_resizing
    # provoque des erreurs CUDA sur modèles quantifiés/fp16)
    model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
    if BIG_MODEL:
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False

    # Si les embeddings d'entrée et de sortie ne sont PAS liés (cas de Mistral), il
    # faut aussi entraîner les lignes du lm_head, sinon les logits des nouveaux
    # tokens restent figés à leur init aléatoire.
    tied = getattr(model.config, "tie_word_embeddings", False)
    trainable_tokens = ({"embed_tokens": NEW_TOKEN_IDS} if tied
                        else {"embed_tokens": NEW_TOKEN_IDS, "lm_head": NEW_TOKEN_IDS})
    lora_cfg = LoraConfig(
        task_type="CAUSAL_LM",
        r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=CONFIG["lora_dropout"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        trainable_token_indices=trainable_tokens,
    )
    model = get_peft_model(model, lora_cfg)
    if not BIG_MODEL:
        model.to(DEVICE)
    return model

# Premier chargement : télécharge les poids si besoin, pour que le jour 1 ne paie
# pas le téléchargement au milieu de la boucle. Le modèle est libéré aussitôt.
import gc
_warm = new_model()
_warm.print_trainable_parameters()
del _warm
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
print("Poids du modèle en cache local ✅ — les 7 jours rechargeront depuis le disque.")

In [ ]:
# ==================== CONTEXTE D'UN JOUR ====================
# Tout ce qui dépend du jour vit ici, et NULLE PART ailleurs. Dans le notebook
# d'origine, `evaluate_cell_accuracy` lisait `users_train`, `GROUP_MODEL`,
# `windows_of_group`… en globales : en boucle, oublier d'en rebinder une seule
# fait valider le mardi sur les utilisateurs du lundi, silencieusement. Ici, la
# fonction reçoit son `ctx` en argument : l'oubli devient impossible.
import random
from cellid_encoding import encode_events, make_chunks, in_transition_window

if USE_GROUPS:
    from user_groups import (fit_groups, derive_group_windows, describe_groups,
                             group_token)


class DayContext:
    """État d'un jour : utilisateurs, groupes de comportement, créneaux, baselines."""

    def __init__(self, day, verbose=True):
        self.day = day
        self.users_train = DAY_DATA[day]["train"]
        self.users_test = DAY_DATA[day]["test"]

        # --- groupes ajustés sur le TRAIN DE CE JOUR uniquement -------------
        # Les utilisateurs de test n'influencent jamais la géométrie des groupes,
        # et un jour n'hérite jamais des centroïdes d'un autre.
        if USE_GROUPS:
            self.group_model = fit_groups(
                self.users_train, n_groups=N_GROUPS,
                bands=[tuple(b) for b in CONFIG["group_bands"]],
                shrinkage=CONFIG["group_shrinkage"], seed=CONFIG["seed"])
            self.oracle_train = self.group_model.assign_all(self.users_train)
            self.oracle_test = self.group_model.assign_all(self.users_test)
            if CONFIG.get("group_windows"):
                self.group_windows = {int(g): [tuple(w) for w in ws]
                                      for g, ws in CONFIG["group_windows"].items()}
            else:
                self.group_windows = derive_group_windows(
                    self.users_train, self.oracle_train, N_GROUPS,
                    min_deficit=CONFIG["group_window_min_deficit"],
                    weight_gain=CONFIG["group_window_weight_gain"],
                    max_weight=CONFIG["group_window_max_weight"])
        else:
            self.group_model = None
            self.group_windows = {}
            self.oracle_train = [None] * len(self.users_train)
            self.oracle_test = [None] * len(self.users_test)

        # Étiquettes utilisées à l'ENTRAÎNEMENT : détectées sur le préfixe de
        # contexte, donc la même distribution qu'à l'évaluation. Entraîner sur
        # l'oracle et tester sur du détecté créerait un décalage inutile.
        self.train_group = [self.detect_group_prefix(cells, hours, CONFIG["context_fraction"])
                            for _, cells, hours in self.users_train]

        # Contexte d'entraînement : chaque utilisateur contribue son préfixe, le
        # reste (jamais entraîné) sert de cible de validation.
        self.context_users = []
        for (uid, cells, hours), grp in zip(self.users_train, self.train_group):
            cut = split_index(len(cells), CONFIG["context_fraction"])
            self.context_users.append(
                (uid, cells[:cut], hours[:cut] if hours is not None else None, grp))

        # Sous-échantillon FIXE de validation (tiré au seed, donc identique d'un
        # jour à l'autre en proportion et reproductible d'une session à l'autre).
        k = WEEK_CONFIG["val_subsample"]
        if k is None or k >= len(self.users_train):
            self.val_users = self.users_train
        else:
            idx = sorted(random.Random(CONFIG["seed"]).sample(range(len(self.users_train)), k))
            self.val_users = [self.users_train[i] for i in idx]

        self.baseline_val = baseline_persistence(self.val_users, CONFIG["context_fraction"])
        self.baseline_test = baseline_persistence(self.users_test, CONFIG["context_fraction"])

        if verbose:
            self.report()

    # ------------------------------------------------------------ groupes
    def detect_group(self, cells, hours):
        """Groupe d'un utilisateur à partir de la tranche de séquence fournie.
        À l'évaluation on ne lui passe QUE le préfixe de contexte : l'étiquette
        ne peut donc pas venir des positions que le modèle doit prédire."""
        if not USE_GROUPS:
            return None
        return self.group_model.assign(cells, hours) if hours is not None else 0

    def detect_group_prefix(self, cells, hours, fraction):
        cut = split_index(len(cells), fraction)
        return self.detect_group(cells[:cut], hours[:cut] if hours is not None else None)

    def windows_of_group(self, group):
        """Créneaux du groupe. Repli sur CONFIG['transition_windows'] hors mode groupe."""
        if not USE_GROUPS or group is None:
            return CONFIG["transition_windows"]
        return self.group_windows.get(group, [])

    # ------------------------------------------------------------ encodage
    def group_prefix_ids(self, group):
        """Préfixe de séquence : le prompt, suivi du token de groupe <Gk>. Placer le
        groupe en tête conditionne TOUTES les prédictions (attention causale)."""
        if not USE_GROUPS or group is None:
            return PREFIX_IDS
        return PREFIX_IDS + [GROUP_TO_ID[group_token(group)]]

    def encode_sequence(self, cells, hours, n_masked_cells=0, group=None):
        """prefix (+ token de groupe) + (heure, cell_id) interleavés ; labels -100 sur
        le préfixe, les heures et les n_masked_cells premiers cell_id. Les poids de
        loss viennent des créneaux DU GROUPE."""
        ids, labels, weights = encode_events(
            cells, hours, CELL_TO_ID, HOUR_TO_ID, self.group_prefix_ids(group),
            self.windows_of_group(group), n_masked_cells=n_masked_cells)
        L = CONFIG["max_seq_len"]
        ids, labels, weights = ids[:L], labels[:L], weights[:L]
        return {"input_ids": ids, "labels": labels, "weights": weights,
                "attention_mask": [1] * len(ids)}

    # ------------------------------------------------------------ rapport
    def group_rows(self):
        """Lignes de synthèse par groupe, sérialisables dans le JSON du jour.
        `describe_groups` ne renvoie que {"group", "n"} pour un groupe VIDE : avec
        4 groupes et une centaine d'utilisateurs de test, un groupe sans aucun
        représentant côté test arrive, et lire ses clés absentes ferait tomber tout
        le jour. D'où les .get() ci-dessous."""
        if not USE_GROUPS:
            return []
        rows = describe_groups(self.users_train, self.oracle_train, self.group_model, N_GROUPS)
        rows_te = describe_groups(self.users_test, self.oracle_test, self.group_model, N_GROUPS)
        nan = float("nan")
        out = []
        for r, rt in zip(rows, rows_te):
            out.append({
                "group": int(r["group"]),
                "n_train": int(r.get("n", 0)), "share_train": float(r.get("share", 0.0)),
                "n_test": int(rt.get("n", 0)), "share_test": float(rt.get("share", 0.0)),
                "profile": [float(v) for v in r.get("profile", [])],
                "persistence": float(r.get("persistence", nan)),
                "mean_len": float(r.get("mean_len", nan)),
                "windows": [[int(s), int(e), float(w)]
                            for s, e, w in self.windows_of_group(r["group"])],
            })
        return out

    def report(self):
        print(f"  {len(self.users_train)} utilisateurs train | {len(self.users_test)} test | "
              f"validation sur {len(self.val_users)} | baseline test {self.baseline_test:.1%}")
        if not USE_GROUPS:
            return
        band_names = [b[0] for b in self.group_model.bands]
        band_hdr = "  ".join(f"{n[:7]:>7}" for n in band_names)
        print(f"  grp  train        test  | {band_hdr} | persist. | créneaux (heure→poids)")
        for r in self.group_rows():
            if not r["n_train"]:
                print(f"  G{r['group']}   (aucun utilisateur d'entraînement)")
                continue
            prof = "  ".join(f"{v:6.0%} " for v in r["profile"])
            wins = ", ".join(f"{s:02d}-{e:02d}h→×{w:.1f}" for s, e, w in r["windows"]) or "aucun"
            print(f"  G{r['group']}  {r['n_train']:4d} ({r['share_train']:.0%})  "
                  f"{r['n_test']:3d} ({r['share_test']:.0%}) | {prof}| "
                  f"{r['persistence']:7.1%}  | {wins}")


class CellDataset(torch.utils.data.Dataset):
    """Fenêtres glissantes sur les préfixes de contexte, loss sur les seuls cell_id."""

    def __init__(self, ctx, users):
        self.items, self.sample_weights, self.groups = [], [], []
        for user in users:
            uid, cells, hours = user[0], user[1], user[2]
            grp = user[3] if len(user) > 3 else None
            wins = ctx.windows_of_group(grp)
            for start, end, n_ctx in make_chunks(len(cells), CONFIG["chunk_len"],
                                                 CONFIG["chunk_stride"]):
                chunk_hours = hours[start:end] if hours is not None else None
                self.items.append(ctx.encode_sequence(cells[start:end], chunk_hours,
                                                      n_masked_cells=n_ctx, group=grp))
                self.groups.append(grp)
                # Poids d'échantillonnage : > 1.0 si le chunk contient au moins une
                # position supervisée dans un créneau de transition de SON groupe.
                has_transition = False
                if chunk_hours is not None:
                    for idx in range(n_ctx, end - start):
                        if in_transition_window(chunk_hours[idx], wins):
                            has_transition = True
                            break
                self.sample_weights.append(
                    CONFIG.get("transition_chunk_oversample", 1.0) if has_transition else 1.0)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids":      torch.tensor([b["input_ids"] + [pad] * (L - len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([b["attention_mask"] + [0] * (L - len(b["attention_mask"])) for b in batch]),
        "labels":         torch.tensor([b["labels"] + [-100] * (L - len(b["labels"])) for b in batch]),
        "weights":        torch.tensor([b["weights"] + [1.0] * (L - len(b["weights"])) for b in batch]),
    }

print("DayContext / CellDataset prêts.")

In [ ]:
# ==================== ÉVALUATION BATCHÉE : accuracy top-k ====================
# Même protocole et mêmes métriques que le notebook d'origine, mais les
# utilisateurs sont regroupés en batches au lieu d'un forward chacun. Le padding
# est à DROITE et l'attention_mask fourni : en attention causale, les positions
# réelles ne voient jamais le padding, donc le score est identique à l'unitaire.
# Les séquences sont triées par longueur pour que chaque batch soit homogène et
# le padding minimal.
from collections import Counter
from cellid_encoding import event_token_positions


@torch.no_grad()
def evaluate_cell_accuracy(model, ctx, users, ks=(1, 3, 5), context_fraction=None,
                           transition_windows=None, group_mode="detected",
                           oracle_groups=None, batch_size=None, per_user=False):
    """Teacher forcing : pour chaque position i>=1, prédit le cell i depuis le contexte
    0..i-1 (toujours le VRAI historique, heures comprises si disponibles).
    Si context_fraction est fourni, ne compte que les positions de la CIBLE.
    Deux variantes top-k : classique (argmax sur les cell_id) et top1_seen (argmax
    restreint aux cellules DÉJÀ VUES dans le préfixe de l'utilisateur).

    group_mode -- comment le token <Gk> est choisi :
      "detected" : détecté sur le SEUL préfixe de contexte (protocole d'inférence réel).
      "oracle"   : détecté sur la séquence complète (borne supérieure, pas un score
                   annonçable : mesure ce que coûte une erreur de détection).
      "none"     : aucun token de groupe (comparaison sans cross-training).
      un entier k: force <Gk> pour tous (ablation « mauvais groupe »).

    per_user=True ajoute le détail par utilisateur (distribution, graphiques) sans
    aucun forward supplémentaire.
    """
    was_training = model.training
    model.eval()
    bs = batch_size or WEEK_CONFIG["eval_batch_size"]
    kmax = max(ks)
    cell_ids_dev = CELL_IDS_T.to(DEVICE)

    # --- 1) préparation : encodage + choix du groupe, par utilisateur --------
    prepared = []
    detect_agree = detect_n = 0
    for ui, (uid, cells, hours) in enumerate(users):
        if len(cells) < 2:
            continue
        min_j = split_index(len(cells), context_fraction) if context_fraction is not None else 1
        if not USE_GROUPS or group_mode == "none":
            grp = None
        elif isinstance(group_mode, int):
            grp = group_mode
        elif group_mode == "oracle":
            grp = (oracle_groups[ui] if oracle_groups is not None
                   else ctx.detect_group(cells, hours))
        else:                                    # "detected" -- préfixe uniquement
            cut = (min_j if context_fraction is not None
                   else split_index(len(cells), CONFIG.get("group_detect_fraction", 0.5)))
            grp = ctx.detect_group(cells[:cut], hours[:cut] if hours is not None else None)
            ref = (oracle_groups[ui] if oracle_groups is not None
                   else ctx.detect_group(cells, hours))
            detect_agree += grp == ref
            detect_n += 1
        wins = transition_windows if transition_windows is not None else ctx.windows_of_group(grp)
        enc = ctx.encode_sequence(cells, hours, group=grp)
        prepared.append({"uid": uid, "cells": cells, "hours": hours, "grp": grp, "wins": wins,
                         "ids": enc["input_ids"], "n_prefix": len(ctx.group_prefix_ids(grp)),
                         "min_j": min_j})

    hits = {k: 0 for k in ks}
    hits_seen = total = 0
    hits_seen_in = tot_in = hits_seen_out = tot_out = 0
    win_hits, win_tots = Counter(), Counter()
    grp_hits, grp_tots = Counter(), Counter()
    per_user_rows = []

    def score_one(p, logits):
        """Boucle de scoring d'UN utilisateur sur ses logits (identique à l'unitaire)."""
        nonlocal hits_seen, total, hits_seen_in, tot_in, hits_seen_out, tot_out
        cells, hours, wins = p["cells"], p["hours"], p["wins"]
        positions = event_token_positions(len(cells), has_hours=hours is not None)
        seen = torch.zeros(len(CELL_TOKEN_IDS), dtype=torch.bool)
        u_hits = {k: 0 for k in ks}
        u_seen = u_tot = 0
        for j in range(len(cells)):
            token_pos = p["n_prefix"] + positions[j]
            if token_pos >= len(p["ids"]):
                break   # séquence tronquée par max_seq_len : rien de plus à évaluer
            target_pos = CELL_POS[CELL_TO_ID[cells[j]]]
            if j == 0:
                seen[target_pos] = True
                continue
            if j >= p["min_j"]:
                scores = logits[token_pos - 1]
                top = scores.topk(kmax).indices.tolist()
                for k in ks:
                    hit = int(target_pos in top[:k])
                    hits[k] += hit
                    u_hits[k] += hit
                masked = scores.clone()
                masked[~seen] = float("-inf")
                hit_seen = int(int(masked.argmax()) == target_pos)
                hits_seen += hit_seen
                u_seen += hit_seen
                total += 1
                u_tot += 1
                if p["grp"] is not None:
                    grp_tots[p["grp"]] += 1
                    grp_hits[p["grp"]] += hit_seen
                if wins and hours is not None:
                    h = hours[j]
                    in_any = False
                    for (start, end, _) in wins:
                        if start <= h < end:
                            win_tots[(start, end)] += 1
                            win_hits[(start, end)] += hit_seen
                            in_any = True
                            break
                    if in_any:
                        tot_in += 1; hits_seen_in += hit_seen
                    else:
                        tot_out += 1; hits_seen_out += hit_seen
            seen[target_pos] = True
        if per_user and u_tot:
            per_user_rows.append({"user_id": p["uid"], "group": p["grp"], "n": u_tot,
                                  "top1": u_hits[1] / u_tot, "top1_seen": u_seen / u_tot})

    # --- 2) forwards batchés, du plus long au plus court --------------------
    order = sorted(range(len(prepared)), key=lambda i: -len(prepared[i]["ids"]))
    pad = tokenizer.pad_token_id
    for b0 in range(0, len(order), bs):
        chunk = [prepared[i] for i in order[b0:b0 + bs]]
        L = max(len(p["ids"]) for p in chunk)
        input_ids = torch.tensor([p["ids"] + [pad] * (L - len(p["ids"])) for p in chunk],
                                 device=DEVICE)
        attn = torch.tensor([[1] * len(p["ids"]) + [0] * (L - len(p["ids"])) for p in chunk],
                            device=DEVICE)
        out = model(input_ids=input_ids, attention_mask=attn).logits
        # On ne garde que les colonnes des cell_id (369 sur ~152 000) et on repasse
        # sur CPU immédiatement : garder les logits complets de tout un batch
        # saturerait la VRAM.
        sel = out[:, :, cell_ids_dev].float().cpu()
        del out
        for row, p in enumerate(chunk):
            score_one(p, sel[row, :len(p["ids"])])
        del sel

    if was_training:
        model.train()

    if total == 0:
        result = {**{f"top{k}": float("nan") for k in ks}, "top1_seen": float("nan"), "n": 0}
    else:
        result = {**{f"top{k}": hits[k] / total for k in ks},
                  "top1_seen": hits_seen / total, "n": total}
    result["top1_seen_in_window"] = hits_seen_in / tot_in if tot_in else float("nan")
    result["top1_seen_out_window"] = hits_seen_out / tot_out if tot_out else float("nan")
    result["n_in_window"] = tot_in
    result["window_breakdown"] = {w: {"acc": win_hits[w] / win_tots[w], "n": win_tots[w]}
                                  for w in sorted(win_tots) if win_tots[w]}
    result["group_breakdown"] = {g: {"acc": grp_hits[g] / grp_tots[g], "n": grp_tots[g]}
                                 for g in sorted(grp_tots) if grp_tots[g]}
    result["group_detect_agreement"] = (detect_agree / detect_n) if detect_n else float("nan")
    if per_user:
        result["per_user"] = per_user_rows
    return result

print("Évaluation batchée prête "
      f"(batch={WEEK_CONFIG['eval_batch_size']}, validation sur "
      f"{WEEK_CONFIG['val_subsample'] or 'tous les'} utilisateurs).")

In [ ]:
# ==================== TRAINER PONDÉRÉ + CALLBACKS ====================
import time
from transformers import Trainer, TrainingArguments, TrainerCallback
import torch.nn.functional as F
from torch.utils.data import WeightedRandomSampler


class WeightedLossTrainer(Trainer):
    """Trainer standard, sauf que la cross-entropy est pondérée par position : les
    cell_id dont l'heure tombe dans un créneau de transition comptent plus fort,
    pour pousser le modèle à raisonner plutôt que recopier le cell_id précédent.
    Sans heures (poids == 1.0 partout), équivalent à la loss standard.
    Accepte aussi un sampler personnalisé (sur-échantillonnage des fenêtres qui
    touchent un créneau)."""

    def __init__(self, *args, sampler=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.custom_sampler = sampler

    def _get_train_sampler(self, *args, **kwargs):
        # *args : la signature du Trainer amont a changé (train_dataset optionnel
        # selon la version de transformers) — on l'absorbe pour rester compatible.
        if self.custom_sampler is not None:
            return self.custom_sampler
        return super()._get_train_sampler(*args, **kwargs)

    def get_train_dataloader(self):
        if self.train_dataset is None:
            raise ValueError("Trainer: training requires a train_dataset.")
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.args.train_batch_size,
            sampler=self._get_train_sampler(),
            collate_fn=self.data_collator,
            drop_last=self.args.dataloader_drop_last,
            num_workers=self.args.dataloader_num_workers,
            pin_memory=self.args.dataloader_pin_memory,
        )

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        weights = inputs.pop("weights")
        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        shift_weights = weights[..., 1:].contiguous()
        losses = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1),
            ignore_index=-100, reduction="none")
        mask = (shift_labels.view(-1) != -100).float()
        weighted = losses * shift_weights.reshape(-1) * mask
        denom = num_items_in_batch if num_items_in_batch is not None else mask.sum().clamp(min=1)
        return ((weighted.sum() / denom, outputs) if return_outputs else weighted.sum() / denom)


class RamGuardCallback(TrainerCallback):
    """Surveille la RAM système ; si elle devient critique : checkpoint puis arrêt
    PROPRE (pas de swap massif ni de gel de la machine)."""

    def __init__(self):
        self.triggered = False

    def on_step_end(self, args, state, control, **kw):
        if state.global_step % CONFIG["ram_check_every_steps"] != 0:
            return
        if DEVICE == "mps":
            torch.mps.empty_cache()
        vm = psutil.virtual_memory()
        if (vm.available / 1e9 < CONFIG["min_free_ram_gb"]
                or vm.percent / 100 > CONFIG["max_ram_used_fraction"]):
            print(f"\n⚠️  GUARD RAM : {vm.percent:.0f}% utilisée, "
                  f"{vm.available / 1e9:.1f} Go libres → arrêt propre de ce jour.")
            self.triggered = True
            control.should_save = True
            control.should_training_stop = True


class AccuracyStopCallback(TrainerCallback):
    """Évalue l'accuracy de validation à chaque epoch, garde les meilleurs poids en
    RAM, et stoppe sur plateau.

    Différence avec le notebook d'origine : rien n'est écrit sur le disque à chaque
    amélioration (~550 Mo × 7 jours de I/O pour rien). Seul l'adaptateur du
    meilleur modèle est sauvegardé, une fois, en fin de jour.
    La validation porte sur ctx.val_users — un sous-échantillon FIXE du train, dont
    seule la partie cible (jamais entraînée) est comptée."""

    def __init__(self, model, ctx):
        self.model, self.ctx = model, ctx
        self.history = []
        self.best = 0.0
        self.best_params = None
        self.best_epoch = None
        self.since_best = 0
        self.reason = None
        self._t0 = time.time()

    def on_epoch_end(self, args, state, control, **kw):
        t0 = time.time()
        m = evaluate_cell_accuracy(self.model, self.ctx, self.ctx.val_users,
                                   context_fraction=CONFIG["context_fraction"])
        score = m["top1_seen"]       # métrique pilote : argmax restreint au répertoire vu
        self.history.append({"epoch": round(state.epoch, 3), "top1": m["top1"],
                             "top1_seen": score, "top3": m["top3"], "top5": m["top5"],
                             "eval_sec": time.time() - t0,
                             "elapsed_sec": time.time() - self._t0})
        print(f"    epoch {state.epoch:5.1f} | val top1={m['top1']:.1%}  top1_seen={score:.1%}  "
              f"top3={m['top3']:.1%}  top5={m['top5']:.1%}  ({time.time() - t0:.0f}s d'éval)")
        if score > self.best:
            self.best, self.since_best = score, 0
            self.best_epoch = round(state.epoch, 3)
            # Copie CPU des poids entraînables : l'éval finale utilise le MEILLEUR
            # modèle, pas celui (sur-appris) de la dernière epoch.
            self.best_params = {n: p.detach().cpu().clone()
                                for n, p in self.model.named_parameters() if p.requires_grad}
        else:
            self.since_best += 1
        if DEVICE == "mps":
            torch.mps.empty_cache()
        if self.since_best >= CONFIG["early_stop_patience"]:
            self.reason = (f"plateau : {CONFIG['early_stop_patience']} epochs sans progrès "
                           f"(meilleur = {self.best:.1%})")
            control.should_training_stop = True


def training_arguments(ckpt_dir):
    """Arguments d'entraînement, identiques pour les 7 jours."""
    if WEEK_CONFIG["train_batch_size"]:
        batch_size = WEEK_CONFIG["train_batch_size"]
        grad_accum = WEEK_CONFIG["grad_accum"] or 1
    elif DEVICE == "cuda":
        # IDENTIQUE au notebook d'origine. Monter le batch pour aller plus vite
        # serait une fausse bonne idée : à learning rate constant, batch 32 donne
        # 4x moins de pas d'optimisation par epoch (65 au lieu de 260 sur un jour
        # de 2075 fenêtres) et change complètement le régime d'apprentissage. Le
        # gain de vitesse doit venir de l'évaluation, pas de l'optimiseur.
        batch_size, grad_accum = (4, 2) if BIG_MODEL else (8, 1)
    elif DEVICE == "mps":
        batch_size, grad_accum = 4, 2
    else:
        batch_size, grad_accum = 1, 8
    return TrainingArguments(
        output_dir=str(ckpt_dir),
        num_train_epochs=CONFIG["max_epochs"],
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        learning_rate=CONFIG["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_steps=20,
        logging_steps=50,
        weight_decay=CONFIG["weight_decay"],
        # Aucun checkpoint périodique : les meilleurs poids vivent en RAM et seul
        # l'adaptateur final est écrit. Le guard RAM peut toujours forcer une
        # sauvegarde d'urgence via control.should_save.
        save_strategy="no",
        save_total_limit=CONFIG["save_total_limit"],
        bf16=(DEVICE == "cuda" and DTYPE == torch.bfloat16),
        fp16=(DEVICE == "cuda" and DTYPE == torch.float16),
        report_to=[],
        seed=CONFIG["seed"],
        dataloader_pin_memory=False,
        remove_unused_columns=False,
        disable_tqdm=False,
    )

print("Trainer et callbacks prêts.")

In [ ]:
# ==================== UN JOUR = UN ENTRAÎNEMENT COMPLET ====================
import gc, time

def _jsonable(obj):
    """Rend un résultat d'évaluation sérialisable : les clés tuple (4, 6) des
    breakdowns deviennent "04-06h", les scalaires numpy deviennent des float."""
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if isinstance(k, tuple) and len(k) == 2:
                k = f"{int(k[0]):02d}-{int(k[1]):02d}h"
            out[str(k)] = _jsonable(v)
        return out
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, (bool, str)) or obj is None:
        return obj
    if hasattr(obj, "item"):          # scalaires numpy / torch
        return obj.item()
    if isinstance(obj, int):
        return int(obj)
    if isinstance(obj, float):
        return float(obj)
    return obj


def sanity_check(model, ctx, train_ds):
    """Détecte AVANT l'entraînement les deux pannes classiques : des ids de tokens
    hors des tables d'embedding (crash CUDA « device-side assert », asynchrone donc
    illisible), et des labels négatifs autres que -100 (IndexError cryptique)."""
    n_embed = model.get_input_embeddings().weight.shape[0]
    out_emb = model.get_output_embeddings()
    n_out = out_emb.weight.shape[0] if out_emb is not None else n_embed
    max_id = max(max(CELL_TOKEN_IDS), max(PREFIX_IDS),
                 *(HOUR_TOKEN_IDS or [0]), *(GROUP_TOKEN_IDS or [0]))
    print(f"  tokenizer: {len(tokenizer)} | embed_tokens: {n_embed} | lm_head: {n_out} | "
          f"id max utilisé: {max_id}")
    assert max_id < n_embed and max_id < n_out, (
        "Ids de tokens hors des tables d'embedding : resize_token_embeddings n'a pas "
        "été appliqué. Relancez la cellule TOKENIZER.")
    bad = {l for item in (train_ds[0], train_ds[1]) for l in item["labels"] if l < 0 and l != -100}
    assert not bad, (f"Labels invalides {bad} — seule la valeur -100 masque la loss.")
    batch = collate([train_ds[0], train_ds[1]])
    inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "weights"}
    loss = model(**inputs).loss
    loss.backward()
    model.zero_grad()
    print(f"  forward/backward OK — loss initiale : {float(loss.detach()):.2f}")


def run_day(day, verbose=True):
    """Entraîne et évalue UN jour, de bout en bout, et renvoie son enregistrement
    de résultats. Aucun état n'est partagé avec les autres jours en dehors du
    tokenizer et du vocabulaire, qui sont volontairement communs."""
    t_start = time.time()
    print(f"\n{'=' * 78}\n  {day.upper()} — préparation\n{'=' * 78}")
    ctx = DayContext(day, verbose=verbose)

    model = new_model()
    train_ds = CellDataset(ctx, ctx.context_users)
    n_touch = sum(1 for w in train_ds.sample_weights if w > 1.0)
    print(f"  {len(train_ds)} fenêtres d'entraînement "
          f"(chunk={CONFIG['chunk_len']}, stride={CONFIG['chunk_stride']})"
          + (f" — {n_touch} en créneau de transition" if HAS_HOURS else ""))
    sanity_check(model, ctx, train_ds)

    # --- entraînement -------------------------------------------------------
    ckpt_dir = Path(CONFIG["output_dir"]) / "checkpoints" / day
    sampler = None
    if HAS_HOURS and any(w > 1.0 for w in train_ds.sample_weights):
        sampler = WeightedRandomSampler(weights=train_ds.sample_weights,
                                        num_samples=len(train_ds), replacement=True)
    ram_cb = RamGuardCallback()
    acc_cb = AccuracyStopCallback(model, ctx)
    trainer = WeightedLossTrainer(model=model, args=training_arguments(ckpt_dir),
                                  train_dataset=train_ds, data_collator=collate,
                                  callbacks=[ram_cb, acc_cb], sampler=sampler)
    print(f"\n  {day.upper()} — entraînement (plafond {CONFIG['max_epochs']} epochs)")
    t_train = time.time()
    trainer.train()
    train_sec = time.time() - t_train
    stop_reason = ("guard RAM déclenché" if ram_cb.triggered
                   else acc_cb.reason or "plafond d'epochs atteint")
    print(f"  Entraînement terminé en {train_sec / 60:.1f} min — {stop_reason}")

    # On évalue le MEILLEUR modèle (pic de validation), pas celui de la dernière epoch.
    if acc_cb.best_params is not None:
        model.load_state_dict(acc_cb.best_params, strict=False)
        print(f"  Meilleur modèle rechargé (val top1_seen = {acc_cb.best:.1%} "
              f"à l'epoch {acc_cb.best_epoch})")

    record = {
        "day": day, "week": WEEK_CONFIG["week"], "status": "trained",
        "files": {k: str(v) for k, v in DAY_FILES[day].items()},
        "data": _jsonable(DAY_STATS[day]),
        "config": {k: CONFIG[k] for k in ("model_name", "seed", "max_epochs",
                                          "early_stop_patience", "learning_rate", "lora_r",
                                          "lora_alpha", "chunk_len", "chunk_stride",
                                          "context_fraction", "use_groups", "n_groups")},
        "vocab_size_shared": len(VOCAB),
        "has_hours": HAS_HOURS,
        # Tracé pour que la boucle ne prenne JAMAIS un tour de chauffe (limit_users
        # réduit) pour un résultat définitif : elle refait le jour si la valeur a
        # changé depuis l'écriture du JSON.
        "limit_users": WEEK_CONFIG["limit_users"],
        "groups": ctx.group_rows(),
        "training": {"history": acc_cb.history, "best_val_top1_seen": acc_cb.best,
                     "best_epoch": acc_cb.best_epoch, "epochs_run": len(acc_cb.history),
                     "stop_reason": stop_reason, "train_sec": train_sec,
                     "baseline_val": ctx.baseline_val,
                     "n_train_windows": len(train_ds)},
    }
    # Écriture intermédiaire : si l'évaluation casse ou la session tombe pendant les
    # ablations, l'entraînement n'est pas perdu pour rien (le jour sera repris, mais
    # on garde la trace de ce qui s'est passé).
    (RESULTS_DIR / f"{day}.json").write_text(json.dumps(_jsonable(record), ensure_ascii=False,
                                                        indent=1), encoding="utf-8")

    # --- évaluation finale sur le test (utilisateurs jamais vus) ------------
    print(f"\n  {day.upper()} — évaluation")
    ref = CONFIG["context_fraction"]
    fracs = [ref] + [f for f in WEEK_CONFIG["test_context_fractions"] if f != ref]
    by_fraction = {}
    for frac in fracs:
        m = evaluate_cell_accuracy(model, ctx, ctx.users_test, context_fraction=frac,
                                   group_mode="detected", oracle_groups=ctx.oracle_test,
                                   per_user=(frac == ref))
        base = baseline_persistence(ctx.users_test, frac)
        if frac == ref:
            record["per_user"] = m.pop("per_user")
        by_fraction[f"{frac:.2f}"] = {"metrics": _jsonable(m), "baseline": base,
                                      "delta": max(m["top1"], m["top1_seen"]) - base}
        tag = "  (référence)" if frac == ref else ""
        print(f"    contexte {frac:.0%}{tag:<13} | n={m['n']:5d} | top1={m['top1']:.1%}  "
              f"top1_seen={m['top1_seen']:.1%}  top3={m['top3']:.1%}  top5={m['top5']:.1%}  "
              f"| baseline {base:.1%}  écart {by_fraction[f'{frac:.2f}']['delta']:+.1%}")
    record["test"] = {"by_fraction": by_fraction, "reference_fraction": f"{ref:.2f}"}

    if HAS_HOURS:
        def test_windows_of(ui):
            _, cells, hours = ctx.users_test[ui]
            return ctx.windows_of_group(ctx.detect_group_prefix(cells, hours, ref))
        bd = baseline_persistence_breakdown(ctx.users_test, ref, test_windows_of)
        record["baseline_breakdown"] = _jsonable(bd)
        m_ref = by_fraction[f"{ref:.2f}"]["metrics"]
        print(f"    créneaux de transition : modèle {m_ref['top1_seen_in_window']:.1%} dedans / "
              f"{m_ref['top1_seen_out_window']:.1%} dehors  |  baseline "
              f"{bd['in_window']:.1%} / {bd['out_window']:.1%}")

    # --- évaluation globale (100% de l'historique, diagnostic) --------------
    if WEEK_CONFIG["run_global_eval"]:
        m_all = evaluate_cell_accuracy(model, ctx, ctx.users_test, context_fraction=None,
                                       group_mode="detected", oracle_groups=ctx.oracle_test)
        base_all = baseline_persistence(ctx.users_test, context_fraction=None)
        record["global_eval"] = {"metrics": _jsonable(m_all), "baseline": base_all}
        print(f"    global (séquence complète)  | n={m_all['n']:5d} | "
              f"top1_seen={m_all['top1_seen']:.1%} | baseline {base_all:.1%}  "
              f"(diagnostic : le groupe y est détecté sur les "
              f"{CONFIG['group_detect_fraction']:.0%} premiers événements, pas un score annonçable)")

    # --- résultats par groupe ------------------------------------------------
    if USE_GROUPS:
        per_group = []
        for g in range(N_GROUPS):
            sub = [u for u, lab in zip(ctx.users_test, ctx.oracle_test) if lab == g]
            if not sub:
                per_group.append({"group": g, "n_users": 0})
                continue
            mg = evaluate_cell_accuracy(model, ctx, sub, context_fraction=ref, group_mode="detected")
            bg = baseline_persistence(sub, ref)
            per_group.append({"group": g, "n_users": len(sub), "n": mg["n"],
                              "top1": mg["top1"], "top1_seen": mg["top1_seen"],
                              "baseline": bg, "delta": max(mg["top1"], mg["top1_seen"]) - bg})
        record["per_group"] = _jsonable(per_group)
        print("    par groupe : " + "  ".join(
            f"G{r['group']}={r['top1_seen']:.1%}({r['delta']:+.1%})"
            for r in per_group if r.get("n_users")))

    # --- ablation GROUPE : le cross-training sert-il ? ----------------------
    if USE_GROUPS and WEEK_CONFIG["run_group_ablation"]:
        variants = [("détecté (protocole réel)", dict(group_mode="detected",
                                                      oracle_groups=ctx.oracle_test)),
                    ("oracle (borne supérieure)", dict(group_mode="oracle",
                                                       oracle_groups=ctx.oracle_test)),
                    ("sans token de groupe", dict(group_mode="none"))]
        variants += [(f"forcé à G{g}", dict(group_mode=g)) for g in range(N_GROUPS)]
        rows, ref_score = [], None
        for label, kw in variants:
            m = evaluate_cell_accuracy(model, ctx, ctx.users_test, context_fraction=ref, **kw)
            if ref_score is None:
                ref_score = m["top1_seen"]
            rows.append({"variant": label, "top1_seen": m["top1_seen"],
                         "delta_vs_detected": m["top1_seen"] - ref_score,
                         "group_detect_agreement": m["group_detect_agreement"]})
        record["ablation_group"] = _jsonable(rows)
        print("    ablation groupe : " + "  ".join(
            f"{r['variant'].split()[0]}={r['top1_seen']:.1%}" for r in rows[:3]))

        # Créneaux du groupe VS ancien créneau global unique, à modèle identique :
        # ne change que le découpage du breakdown dans/hors créneaux.
        if HAS_HOURS:
            split_rows = []
            for label, wins in (("créneaux par groupe", None),
                                ("créneau global unique", CONFIG["transition_windows"])):
                m = evaluate_cell_accuracy(model, ctx, ctx.users_test, context_fraction=ref,
                                           group_mode="detected", oracle_groups=ctx.oracle_test,
                                           transition_windows=wins)
                split_rows.append({"split": label, "n_in_window": m["n_in_window"],
                                   "in_window": m["top1_seen_in_window"],
                                   "out_window": m["top1_seen_out_window"],
                                   "gap": m["top1_seen_in_window"] - m["top1_seen_out_window"]})
            record["window_split"] = _jsonable(split_rows)

    # --- ablation HEURE : le modèle utilise-t-il l'heure ? ------------------
    if HAS_HOURS and WEEK_CONFIG["run_hour_ablation"]:
        def corrupt_hours(users, mode, seed=CONFIG["seed"]):
            """Mêmes cell_id, mêmes longueurs, seules les heures changent."""
            rng_local = random.Random(seed)
            out = []
            for uid, cells, hours in users:
                if hours is None:
                    out.append((uid, cells, hours)); continue
                if mode == "shuffle":
                    h = list(hours); rng_local.shuffle(h)
                elif mode == "fixed":
                    h = [12] * len(hours)
                elif mode == "shift12":
                    h = [(hh + 12) % 24 for hh in hours]
                else:
                    raise ValueError(mode)
                out.append((uid, cells, h))
            return out

        # Le groupe est gardé FIXE (oracle) d'une variante à l'autre : sinon corrompre
        # les heures changerait aussi le groupe détecté, et on ne saurait plus ce qu'on
        # mesure.
        hour_rows = []
        for label, short, variant in (
                ("heures réelles", "réelles", ctx.users_test),
                ("heures mélangées", "mélangées", corrupt_hours(ctx.users_test, "shuffle")),
                ("heure fixe (12h)", "fixe", corrupt_hours(ctx.users_test, "fixed")),
                ("heures décalées +12h", "décalées", corrupt_hours(ctx.users_test, "shift12"))):
            m = evaluate_cell_accuracy(model, ctx, variant, context_fraction=ref,
                                       group_mode="oracle", oracle_groups=ctx.oracle_test)
            hour_rows.append({"variant": label, "short": short, "top1_seen": m["top1_seen"],
                              "in_window": m["top1_seen_in_window"],
                              "out_window": m["top1_seen_out_window"]})
        record["ablation_hour"] = _jsonable(hour_rows)
        print("    ablation heure : " + "  ".join(
            f"{r['short']}={r['top1_seen']:.1%}" for r in hour_rows))

    # --- sauvegarde ---------------------------------------------------------
    if WEEK_CONFIG["save_adapters"]:
        adapter_dir = RESULTS_DIR / "adapters" / day
        model.save_pretrained(adapter_dir)
        record["adapter_dir"] = str(adapter_dir)

    record["status"] = "complete"
    record["runtime_sec"] = time.time() - t_start
    (RESULTS_DIR / f"{day}.json").write_text(json.dumps(_jsonable(record), ensure_ascii=False,
                                                        indent=1), encoding="utf-8")
    print(f"  ✅ {day} terminé en {record['runtime_sec'] / 60:.1f} min "
          f"→ {RESULTS_DIR / (day + '.json')}")

    # --- libération : sans ça, la VRAM du jour N-1 fait échouer le jour N ---
    del trainer, model, train_ds, ctx, acc_cb
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()
    return record

print("run_day() prêt.")

In [ ]:
# ==================== BOUCLE SUR LA SEMAINE (REPRENABLE) ====================
# Cette cellule peut être relancée telle quelle après une déconnexion Colab : les
# jours déjà terminés sont relus depuis leur JSON et sautés. Pour refaire un jour,
# mettez-le dans WEEK_CONFIG["force_rerun"] (ou supprimez son JSON).
import time

RESULTS = {}
t_week = time.time()
done, todo, stale = [], [], []
for day in WEEK_CONFIG["days"]:
    path = RESULTS_DIR / f"{day}.json"
    if path.exists() and day not in WEEK_CONFIG["force_rerun"]:
        rec = json.loads(path.read_text(encoding="utf-8"))
        if rec.get("status") == "complete":
            # Un résultat produit avec un AUTRE limit_users (typiquement un tour de
            # chauffe à 60 utilisateurs) n'est pas le résultat demandé : on le refait
            # au lieu de le compter comme terminé et de le mélanger aux autres.
            if rec.get("limit_users") != WEEK_CONFIG["limit_users"]:
                stale.append((day, rec.get("limit_users")))
            else:
                RESULTS[day] = rec
                done.append(day)
                continue
    todo.append(day)

if done:
    print(f"Déjà terminés, sautés : {', '.join(done)}")
if stale:
    print("Résultats obsolètes (produits avec un autre limit_users), refaits : "
          + ", ".join(f"{d} (limit_users={lu})" for d, lu in stale))
print(f"À entraîner ({len(todo)}) : {', '.join(todo) if todo else 'aucun — la semaine est complète'}\n")

for i, day in enumerate(todo, 1):
    print(f"\n########## {i}/{len(todo)} : {day} ##########")
    RESULTS[day] = run_day(day)
    elapsed = time.time() - t_week
    if i < len(todo):
        eta = elapsed / i * (len(todo) - i)
        print(f"\n  ⏱  {elapsed / 60:.0f} min écoulées, ~{eta / 60:.0f} min restantes "
              f"pour les {len(todo) - i} jours suivants")

def load_results(results_dir=None):
    """Les jours terminés, relus depuis le disque. Un jour produit avec un autre
    `limit_users` est ignoré : mélanger un tour de chauffe et un vrai run dans le
    même tableau donnerait une comparaison fausse et parfaitement crédible."""
    d = Path(results_dir) if results_dir else RESULTS_DIR
    out, ignored = {}, []
    for day in WEEK_CONFIG["days"]:
        p = d / f"{day}.json"
        if not p.exists():
            continue
        rec = json.loads(p.read_text(encoding="utf-8"))
        if rec.get("status") != "complete":
            continue
        if rec.get("limit_users") != WEEK_CONFIG["limit_users"]:
            ignored.append(day)
            continue
        out[day] = rec
    if ignored:
        print(f"⚠️  Ignorés (limit_users différent de {WEEK_CONFIG['limit_users']}) : "
              + ", ".join(ignored))
    return out

# Relecture depuis le disque : la synthèse travaille sur ce qui est RÉELLEMENT
# écrit, pas sur des variables en mémoire — donc elle donne le même résultat sur
# une session neuve qui ne ferait que lire les JSON.
RESULTS = load_results()
print(f"\n{len(RESULTS)}/{len(WEEK_CONFIG['days'])} jours disponibles pour la comparaison : "
      f"{', '.join(RESULTS)}")

In [ ]:
# ==================== SYNTHÈSE : LE TABLEAU DE COMPARAISON ====================
# Cette cellule et les suivantes ne lisent QUE les JSON écrits par la boucle :
# elles fonctionnent sur une session neuve, sans GPU, sans refaire un entraînement.
import pandas as pd

RESULTS = load_results()   # défini dans la cellule BOUCLE
if not RESULTS:
    raise RuntimeError(f"Aucun résultat dans {RESULTS_DIR} — lancez d'abord la cellule BOUCLE.")

REF_KEY = f"{CONFIG['context_fraction']:.2f}"
FR_DAYS = {"monday": "lundi", "tuesday": "mardi", "wednesday": "mercredi",
           "thursday": "jeudi", "friday": "vendredi", "saturday": "samedi",
           "sunday": "dimanche"}

rows = []
for day, r in RESULTS.items():
    ref = r["test"]["by_fraction"][REF_KEY]
    m = ref["metrics"]
    tr = r["training"]
    rows.append({
        "jour": FR_DAYS.get(day, day),
        "day": day,
        "n_train": r["data"]["n_train"],
        "n_test": r["data"]["n_test"],
        "événements": r["data"]["n_events"],
        "epochs": tr["epochs_run"],
        "best_ep": tr["best_epoch"] if tr["best_epoch"] is not None else float("nan"),
        "val_top1_seen": tr["best_val_top1_seen"],
        "top1": m["top1"],
        "top3": m["top3"],
        "top5": m["top5"],
        "top1_seen": m["top1_seen"],
        "baseline": ref["baseline"],
        "écart": ref["delta"],
        "dans_créneau": m.get("top1_seen_in_window", float("nan")),
        "hors_créneau": m.get("top1_seen_out_window", float("nan")),
        "détection_groupe": m.get("group_detect_agreement", float("nan")),
        "durée_min": r["runtime_sec"] / 60,
        "arrêt": tr["stop_reason"],
    })
SUMMARY = pd.DataFrame(rows).set_index("jour")

pd.set_option("display.width", 200)
pct = ["val_top1_seen", "top1", "top3", "top5", "top1_seen", "baseline", "écart",
       "dans_créneau", "hors_créneau", "détection_groupe"]
show = SUMMARY[["n_test", "epochs", "best_ep", "top1_seen", "baseline", "écart",
                "top1", "top3", "top5", "dans_créneau", "hors_créneau", "durée_min"]].copy()
print(f"RÉFÉRENCE : contexte {CONFIG['context_fraction']:.0%}, "
      f"utilisateurs de test jamais vus, meilleur modèle de chaque jour\n")
print(show.to_string(formatters={
    **{c: "{:.1%}".format for c in pct if c in show.columns},
    "durée_min": "{:.0f}".format, "best_ep": "{:.0f}".format}))

# --- la lecture qui compte ---------------------------------------------------
best = SUMMARY["écart"].idxmax()
worst = SUMMARY["écart"].idxmin()
spread = SUMMARY["écart"].max() - SUMMARY["écart"].min()
print(f"\nMeilleur jour (écart à SA baseline) : {best} {SUMMARY.loc[best, 'écart']:+.1%}")
print(f"Pire jour                            : {worst} {SUMMARY.loc[worst, 'écart']:+.1%}")
print(f"Amplitude entre jours                : {spread:.1%}")
print(f"Écart moyen sur la semaine           : {SUMMARY['écart'].mean():+.1%} "
      f"(écart-type {SUMMARY['écart'].std():.1%})")
print("\n⚠️  Les 7 jeux n'ont AUCUN utilisateur en commun : un écart entre deux jours mélange")
print("    effet « jour de la semaine » et effet « échantillon d'utilisateurs ». La colonne")
print("    `écart` neutralise l'essentiel de la difficulté propre à l'échantillon, mais pas")
print("    tout. Pour savoir si une différence est réelle, refaites la semaine 2 (w2) et")
print("    comparez : l'écart lundi_w1 vs lundi_w2 EST votre plancher de bruit.")

csv_path = RESULTS_DIR / "week_summary.csv"
SUMMARY.to_csv(csv_path)
print(f"\nTableau écrit dans {csv_path}")

In [ ]:
# ==================== DÉTAILS : GROUPES, ABLATIONS, DISTRIBUTIONS ====================
# Le détail par jour, pour la présentation : ce que chaque jour dit du modèle,
# au-delà du score global.

# --- 1. accuracy par groupe de comportement ---------------------------------
if any(r.get("per_group") for r in RESULTS.values()):
    print("ACCURACY PAR GROUPE (top1_seen, contexte de référence ; écart = gain sur la")
    print("baseline DU GROUPE). Les groupes sont réajustés sur le train de chaque jour :")
    print("G0 reste « le plus sédentaire », mais les effectifs bougent d'un jour à l'autre.\n")
    grp_rows = []
    for day, r in RESULTS.items():
        row = {"jour": FR_DAYS.get(day, day)}
        for g in r.get("per_group", []):
            if g.get("n_users"):
                row[f"G{g['group']}"] = g["top1_seen"]
                row[f"G{g['group']} écart"] = g["delta"]
                row[f"G{g['group']} n"] = g["n_users"]
        grp_rows.append(row)
    GROUPS_DF = pd.DataFrame(grp_rows).set_index("jour")
    cols_acc = [c for c in GROUPS_DF.columns if c.startswith("G") and c[-1].isdigit()]
    cols_dlt = [c for c in GROUPS_DF.columns if c.endswith("écart")]
    print(GROUPS_DF[cols_acc + cols_dlt].to_string(
        formatters={c: "{:.1%}".format for c in cols_acc + cols_dlt}))
    GROUPS_DF.to_csv(RESULTS_DIR / "week_par_groupe.csv")
else:
    GROUPS_DF = pd.DataFrame()

# --- 2. ablation groupe : le cross-training sert-il, chaque jour ? ----------
if any(r.get("ablation_group") for r in RESULTS.values()):
    print("\n\nABLATION GROUPE — top1_seen selon le conditionnement (contexte de référence)")
    print("« détecté » proche de « oracle » = la détection sur préfixe suffit.")
    print("« détecté » nettement au-dessus de « sans token » = le cross-training apporte.")
    print("Un groupe FORCÉ qui fait chuter le score confirme que le modèle lit le token.\n")
    abl_rows = []
    for day, r in RESULTS.items():
        row = {"jour": FR_DAYS.get(day, day)}
        for v in r.get("ablation_group", []):
            row[v["variant"]] = v["top1_seen"]
        abl_rows.append(row)
    ABL_GROUP_DF = pd.DataFrame(abl_rows).set_index("jour")
    print(ABL_GROUP_DF.to_string(formatters={c: "{:.1%}".format for c in ABL_GROUP_DF.columns}))
    gain = (ABL_GROUP_DF["détecté (protocole réel)"] - ABL_GROUP_DF["sans token de groupe"])
    print(f"\nGain du token de groupe : {gain.mean():+.1%} en moyenne "
          f"(min {gain.min():+.1%} le {gain.idxmin()}, max {gain.max():+.1%} le {gain.idxmax()})")
    ABL_GROUP_DF.to_csv(RESULTS_DIR / "week_ablation_groupe.csv")
else:
    ABL_GROUP_DF = pd.DataFrame()

# --- 3. ablation heure : le modèle lit-il l'heure, chaque jour ? ------------
if any(r.get("ablation_hour") for r in RESULTS.values()):
    print("\n\nABLATION HEURE — mêmes utilisateurs, mêmes cell_id, même groupe : seule")
    print("l'heure fournie change. Si « mélangées » et « fixe » restent au niveau des")
    print("heures réelles, le modèle n'exploite pas le signal horaire.\n")
    hour_rows = []
    for day, r in RESULTS.items():
        row = {"jour": FR_DAYS.get(day, day)}
        for v in r.get("ablation_hour", []):
            row[v["variant"]] = v["top1_seen"]
        hour_rows.append(row)
    ABL_HOUR_DF = pd.DataFrame(hour_rows).set_index("jour")
    print(ABL_HOUR_DF.to_string(formatters={c: "{:.1%}".format for c in ABL_HOUR_DF.columns}))
    if "heures réelles" in ABL_HOUR_DF and "heure fixe (12h)" in ABL_HOUR_DF:
        cost = ABL_HOUR_DF["heures réelles"] - ABL_HOUR_DF["heure fixe (12h)"]
        print(f"\nCoût de la suppression de l'heure : {cost.mean():+.1%} en moyenne")
    ABL_HOUR_DF.to_csv(RESULTS_DIR / "week_ablation_heure.csv")
else:
    ABL_HOUR_DF = pd.DataFrame()

# --- 4. distribution par utilisateur ----------------------------------------
# Le score global d'un jour est une moyenne : deux jours de même moyenne peuvent
# cacher des distributions très différentes (beaucoup d'utilisateurs faciles vs
# une population homogène). C'est souvent l'écart le plus parlant à présenter.
PER_USER = {day: pd.DataFrame(r["per_user"]) for day, r in RESULTS.items() if r.get("per_user")}
if PER_USER:
    print("\n\nDISTRIBUTION PAR UTILISATEUR (top1_seen, contexte de référence)\n")
    dist_rows = []
    for day, df in PER_USER.items():
        q = df["top1_seen"].quantile([0.1, 0.25, 0.5, 0.75, 0.9])
        dist_rows.append({"jour": FR_DAYS.get(day, day), "utilisateurs": len(df),
                          "moyenne": df["top1_seen"].mean(), "d1": q[0.1], "q1": q[0.25],
                          "médiane": q[0.5], "q3": q[0.75], "d9": q[0.9],
                          "écart-type": df["top1_seen"].std()})
    DIST_DF = pd.DataFrame(dist_rows).set_index("jour")
    print(DIST_DF.to_string(formatters={c: "{:.1%}".format for c in DIST_DF.columns
                                        if c != "utilisateurs"}))
    DIST_DF.to_csv(RESULTS_DIR / "week_distribution.csv")
else:
    DIST_DF = pd.DataFrame()

print(f"\n\nTous les CSV sont dans {RESULTS_DIR}")

In [ ]:
# ==================== GRAPHIQUES ====================
import matplotlib.pyplot as plt
import numpy as np

# Palette validée (contraste, séparation daltonisme sur paires adjacentes).
# Les couleurs portent une IDENTITÉ (le jour) ou une MAGNITUDE (une seule série) —
# jamais un rang, jamais un dégradé arc-en-ciel.
SURFACE   = "#fcfcfb"
INK       = "#0b0b0b"
INK_SOFT  = "#52514e"
INK_MUTED = "#8a8984"
GRID      = "#e6e5e1"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
BLUE, ORANGE = SERIES[0], SERIES[1]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK_SOFT, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "axes.edgecolor": GRID, "grid.color": GRID,
    "font.size": 10, "axes.titlesize": 12, "figure.dpi": 110,
})

def _clean(ax, ygrid=True):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    if ygrid:
        ax.grid(axis="y", alpha=0.6, linewidth=0.8)
        ax.set_axisbelow(True)

DAYS_PLOT = [d for d in WEEK_CONFIG["days"] if d in RESULTS]
LABELS = [FR_DAYS.get(d, d)[:3] for d in DAYS_PLOT]
COLOR_OF_DAY = {d: SERIES[i % len(SERIES)] for i, d in enumerate(WEEK_CONFIG["days"])}

# ---------------------------------------------------------------- 1. le score
# Une seule série de magnitude (le modèle) -> une seule teinte. La baseline du
# jour est un REPÈRE, pas une série concurrente : trait neutre posé sur la barre.
fig, ax = plt.subplots(figsize=(9, 4.6))
x = np.arange(len(DAYS_PLOT))
top1_seen = [RESULTS[d]["test"]["by_fraction"][REF_KEY]["metrics"]["top1_seen"] for d in DAYS_PLOT]
base = [RESULTS[d]["test"]["by_fraction"][REF_KEY]["baseline"] for d in DAYS_PLOT]

bars = ax.bar(x, top1_seen, width=0.62, color=BLUE, zorder=2)
for xi, b in zip(x, base):
    ax.plot([xi - 0.34, xi + 0.34], [b, b], color=INK_SOFT, linewidth=2, zorder=3,
            solid_capstyle="butt")
for xi, t, b in zip(x, top1_seen, base):
    ax.annotate(f"{t:.1%}", xy=(xi, t), xytext=(0, 5), textcoords="offset points",
                ha="center", fontsize=9.5, fontweight="bold", color=INK)
    ax.annotate(f"{t - b:+.1f} pt", xy=(xi, b), xytext=(0, -14), textcoords="offset points",
                ha="center", fontsize=9, color=INK_SOFT)
ax.set_xticks(x, LABELS)
ax.set_ylabel("top-1 répertoire (test)")
ax.set_ylim(0, max(top1_seen) * 1.25)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Accuracy par jour, et gain sur la baseline du jour")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=BLUE),
                   plt.Line2D([0], [0], color=INK_SOFT, linewidth=2)],
          labels=["modèle (top1_seen)", "baseline persistance du jour"],
          frameon=False, loc="upper left", bbox_to_anchor=(0, 1.0), fontsize=9)
_clean(ax)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------- 2. l'écart
# Le vrai objet de la comparaison : polarité autour de la baseline -> divergent.
fig, ax = plt.subplots(figsize=(9, 3.8))
delta = [RESULTS[d]["test"]["by_fraction"][REF_KEY]["delta"] for d in DAYS_PLOT]
ax.bar(x, delta, width=0.62, color=[BLUE if v >= 0 else "#e34948" for v in delta], zorder=2)
ax.axhline(0, color=INK_SOFT, linewidth=1)
mean_delta = float(np.mean(delta))
ax.axhline(mean_delta, color=INK_MUTED, linestyle="--", linewidth=1)
ax.annotate(f"moyenne {mean_delta:+.1%}", xy=(len(x) - 0.5, mean_delta), xytext=(4, 0),
            textcoords="offset points", fontsize=9, color=INK_SOFT, va="center")
for xi, v in zip(x, delta):
    ax.annotate(f"{v:+.1%}", xy=(xi, v), xytext=(0, 4 if v >= 0 else -13),
                textcoords="offset points", ha="center", fontsize=9.5, color=INK)
ax.set_xticks(x, LABELS)
ax.set_ylabel("modèle − baseline")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:+.0%}")
ax.set_title("Gain du modèle sur « répéter le dernier cell_id », par jour")
_clean(ax)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------- 3. courbes
# Sept identités -> sept teintes de la palette, dans l'ordre fixe, + légende.
fig, ax = plt.subplots(figsize=(9, 4.8))
for d in DAYS_PLOT:
    h = RESULTS[d]["training"]["history"]
    if not h:
        continue
    ax.plot([e["epoch"] for e in h], [e["top1_seen"] for e in h],
            color=COLOR_OF_DAY[d], linewidth=2, marker="o", markersize=4,
            label=FR_DAYS.get(d, d))
    b = RESULTS[d]["training"]["best_epoch"]
    if b is not None:
        best_v = max(e["top1_seen"] for e in h)
        ax.plot([b], [best_v], marker="o", markersize=8, markerfacecolor="none",
                markeredgecolor=COLOR_OF_DAY[d], markeredgewidth=1.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("top-1 répertoire (validation)")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Convergence : validation par epoch (cercle = pic retenu)")
ax.legend(frameon=False, ncol=4, fontsize=9)
_clean(ax)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------------- 4. distribution
if PER_USER:
    fig, ax = plt.subplots(figsize=(9, 4.4))
    data = [PER_USER[d]["top1_seen"].values for d in DAYS_PLOT if d in PER_USER]
    labs = [FR_DAYS.get(d, d)[:3] for d in DAYS_PLOT if d in PER_USER]
    bp = ax.boxplot(data, tick_labels=labs, patch_artist=True, widths=0.55,
                    medianprops=dict(color=INK, linewidth=1.8),
                    whiskerprops=dict(color=INK_MUTED), capprops=dict(color=INK_MUTED),
                    flierprops=dict(marker="o", markersize=3, markerfacecolor=INK_MUTED,
                                    markeredgecolor="none", alpha=0.45))
    for patch in bp["boxes"]:
        patch.set_facecolor(BLUE)
        patch.set_alpha(0.22)
        patch.set_edgecolor(BLUE)
        patch.set_linewidth(1.5)
    ax.set_ylabel("top-1 répertoire, par utilisateur")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_title("Distribution par utilisateur — deux jours de même moyenne\npeuvent cacher "
                 "deux populations différentes", fontsize=11)
    _clean(ax)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------- 5. créneaux
in_w = [RESULTS[d]["test"]["by_fraction"][REF_KEY]["metrics"].get("top1_seen_in_window")
        for d in DAYS_PLOT]
if any(v is not None and not np.isnan(v) for v in in_w):
    out_w = [RESULTS[d]["test"]["by_fraction"][REF_KEY]["metrics"]["top1_seen_out_window"]
             for d in DAYS_PLOT]
    fig, ax = plt.subplots(figsize=(9, 4.2))
    w = 0.36
    ax.bar(x - w / 2, in_w, width=w, color=BLUE, label="dans un créneau de transition", zorder=2)
    ax.bar(x + w / 2, out_w, width=w, color=ORANGE, label="hors créneau", zorder=2)
    ax.set_xticks(x, LABELS)
    ax.set_ylabel("top-1 répertoire (test)")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_title("Les créneaux de transition isolent-ils bien les positions difficiles ?")
    ax.legend(frameon=False, fontsize=9)
    _clean(ax)
    plt.tight_layout()
    plt.show()
    print("Lecture : « dans » nettement SOUS « hors » = le créneau cible bien les positions")
    print("où le modèle doit raisonner. « dans » au-dessus = le créneau est mal placé pour")
    print("ce jour-là, et le surpondérer pousse du gradient là où la baseline a déjà raison.")

# ---------------------------------------------------------------- 6. ablations
if not ABL_GROUP_DF.empty and "sans token de groupe" in ABL_GROUP_DF:
    fig, ax = plt.subplots(figsize=(9, 3.8))
    gain = (ABL_GROUP_DF["détecté (protocole réel)"]
            - ABL_GROUP_DF["sans token de groupe"]).reindex(
                [FR_DAYS.get(d, d) for d in DAYS_PLOT])
    ax.bar(x, gain.values, width=0.62,
           color=[BLUE if v >= 0 else "#e34948" for v in gain.values], zorder=2)
    ax.axhline(0, color=INK_SOFT, linewidth=1)
    for xi, v in zip(x, gain.values):
        ax.annotate(f"{v:+.1%}", xy=(xi, v), xytext=(0, 4 if v >= 0 else -13),
                    textcoords="offset points", ha="center", fontsize=9.5, color=INK)
    ax.set_xticks(x, LABELS)
    ax.set_ylabel("avec token − sans token")
    ax.yaxis.set_major_formatter(lambda v, _: f"{v:+.0%}")
    ax.set_title("Apport du cross-training par groupe, par jour")
    _clean(ax)
    plt.tight_layout()
    plt.show()

## Notes

### Ce que ce notebook garantit sur l'isolation des 7 entraînements

| Risque | Traitement |
|---|---|
| Adaptateurs LoRA empilés | `new_model()` recharge la base et l'enveloppe à neuf. `get_peft_model` modifie le modèle **en place** : le rappeler sur un modèle déjà enveloppé ferait démarrer le jour N sur les poids du jour N−1. |
| Embeddings des nouveaux tokens conservés | `trainable_token_indices` les fait vivre dans l'adaptateur PEFT — ils repartent donc de zéro avec lui. |
| Données du jour précédent | Aucune globale de données : `evaluate_cell_accuracy` et `CellDataset` reçoivent un `DayContext` explicite. |
| Early stopping faussé | `AccuracyStopCallback` est instancié par jour ; son `best` ne traverse pas les jours. |
| Bruit d'initialisation pris pour un effet jour | `set_seed(CONFIG["seed"])` au début de `new_model()` : même init LoRA, même ordre de sampler pour les 7 jours. |
| VRAM saturée au 3ᵉ jour | `del` + `gc.collect()` + `empty_cache()` en fin de `run_day`. |

### Ce qui est partagé, volontairement

Le tokenizer, le vocabulaire `cell_id` (union des 7 jours) et les ids de tokens.
Sans ce partage, un jour au répertoire plus large affronterait un softmax plus
grand, et les `top-k` ne seraient plus comparables. Sur ce jeu le partage est
quasi gratuit : 369 `cell_id` par jour, 369 en union.

### Les groupes sont réajustés par jour

`fit_groups` tourne sur le train **de chaque jour** — un groupe doit rester
dérivable des seules données disponibles ce jour-là. Les groupes étant numérotés
par mobilité croissante, `G0` reste « le plus sédentaire » d'un jour à l'autre,
mais les effectifs et les créneaux dérivés bougent. Ne comparez pas les parts de
groupe au point de pourcentage près entre deux jours.

### Où passe le temps d'une journée

Mesuré sur lundi (1900 utilisateurs, 2075 fenêtres, ~13 epochs avant l'arrêt) :

| Poste | Part de la journée |
|---|---|
| Entraînement (784k tokens/epoch, forward + backward) | **73 %** |
| Validation par epoch (297k tokens/epoch, forward seul) | **27 %** |
| Bloc d'évaluation finale : 4 fractions, global, par groupe, ablations heure et groupe | **~2 %** |

La conséquence pratique : **ne coupez pas les ablations pour gagner du temps**.
Elles portent l'essentiel de la matière de présentation et ne coûtent presque
rien. Le temps est dans les 13 epochs d'entraînement, et accessoirement dans la
validation.

### D'où vient le gain de vitesse

| Changement | Gain | Ce qu'on perd |
|---|---|---|
| Évaluation batchée (batch de 8 au lieu d'un forward par utilisateur) | ~8× sur **toutes** les évaluations : validation par epoch, test, ablations | Rien. Padding à droite + `attention_mask` : en attention causale les positions réelles ne voient jamais le padding, le score est identique à l'unitaire. |
| Modèle de base chargé depuis le cache HF au lieu d'être retéléchargé | le téléchargement n'est payé qu'une fois pour les 7 jours | Rien. |
| `save_strategy="no"` | ~550 Mo d'écriture disque par amélioration de validation | Un jour interrompu **pendant** son entraînement est repris depuis le début. Un jour est atomique ; à ~10-25 min le jour, c'est le bon grain. |
| Détail par utilisateur extrait de la passe d'évaluation de référence | le notebook d'origine refaisait un forward par utilisateur × 7 fractions (700 forwards) | Rien : les mêmes logits servent aux deux usages. |

### Ce qu'il ne faut PAS régler pour aller plus vite

Ces trois réglages sont tentants et faux. Ils ne raccourcissent pas
l'entraînement : ils en changent le **régime**, et un jour ainsi réglé n'est
comparable ni aux autres jours, ni aux runs précédents.

| Fausse bonne idée | Ce qui se passe vraiment |
|---|---|
| Monter le batch d'entraînement (8 → 32) | À learning rate constant, 4× moins de pas d'optimisation par epoch : 65 au lieu de 260 sur un jour de 2075 fenêtres. Et `warmup_steps=20` passe de 8 % à 31 % du premier tiers d'entraînement. Le modèle apprend autre chose, pas la même chose plus vite. |
| Baisser `max_epochs` (20 → 12) | `max_epochs` est l'**horizon du scheduler cosine**, pas seulement un plafond : le passer à 12 fait décroître le learning rate presque deux fois plus vite, donc change la trajectoire dès la première epoch. Comme le pic arrive vers l'epoch 9 et que la patience coupe vers 13, le plafond de 20 ne coûte de toute façon presque rien. |
| Baisser la patience (4 → 3) | Économise au mieux une epoch, et peut couper un jour juste avant son pic. Le gain est dans le bruit, le risque non. |

Les deux réglages de vitesse qui restent **exposés**, et leur contrepartie :

- `WEEK_CONFIG["val_subsample"] = 400` (défaut) fait tomber la validation de 27 %
  à 7 % d'une epoch, soit ~20 % de temps gagné. L'early stopping repose alors sur
  ~6k prédictions au lieu de ~30k, donc l'epoch retenue peut se décaler d'une.
  Mettre `None` pour retrouver le protocole exact du notebook d'origine.
- `WEEK_CONFIG["limit_users"] = 950` divise la journée par deux, entraînement
  compris. C'est le seul levier qui touche vraiment au poste dominant, et il
  change le modèle : à appliquer aux 7 jours ou à aucun.

Dans les deux cas, la règle est la même : le réglage vaut **pour les sept jours à
la fois**, jamais pour un seul. Un jour réglé différemment des autres ne se
compare plus à eux, et c'est précisément ce que ce notebook existe pour éviter.

### Reprise après une déconnexion Colab

Relancer la cellule BOUCLE, c'est tout. Les jours dont le JSON porte
`"status": "complete"` sont relus et sautés. Un jour interrompu pendant son
entraînement a un JSON `"status": "trained"` ou pas de JSON du tout : il est
refait. Pour forcer la reprise d'un jour terminé, mettez son nom dans
`WEEK_CONFIG["force_rerun"]`.

Les cellules SYNTHÈSE, DÉTAILS et GRAPHIQUES ne lisent que les JSON : elles
tournent sur une session neuve, sans GPU, tant que `RESULTS_DIR` pointe sur le
dossier de résultats (Drive).

### La limite méthodologique à annoncer

Les sept jeux n'ont **aucun utilisateur en commun**. Un écart entre deux jours
mélange donc effet « jour de la semaine » et effet « échantillon ». La colonne
`écart` (modèle − baseline du jour) neutralise l'essentiel de la difficulté propre
à l'échantillon, mais pas tout.

Pour trancher : refaire la semaine avec les fichiers `w2` (changer
`WEEK_CONFIG["week"]` et `RESULTS_DIR`), puis comparer. L'écart lundi_w1 vs
lundi_w2 est le **plancher de bruit** : toute différence entre jours inférieure à
ce plancher n'est pas interprétable. C'est la mesure qui rend le classement des
jours défendable en présentation.